# Android/Mobile Cryptography Benchmark Analysis

This notebook loads benchmark CSV files from the results folder, cleans and validates the data, converts units, aggregates results, creates publication-ready plots, computes a trade-off score, and exports summary tables for an MSc Data Science final project.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

ROOT_DIR = Path(r'c:\Tese\results')
INPUT_FOLDER = ROOT_DIR / 'run_results'
OUTPUT_DIR = ROOT_DIR / 'benchmark_analysis_outputs'
PLOTS_DIR = OUTPUT_DIR / 'plots'
TABLES_DIR = OUTPUT_DIR / 'tables'

ALPHA = 0.5
EXPECTED_COLUMNS = [
    'size_bytes', 'round_index', 'enc_ns', 'dec_ns',
    'enc_mem_bytes', 'dec_mem_bytes', 'energy_mWh', 'method'
]

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.figsize': (12, 7),
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
})

for folder in [OUTPUT_DIR, PLOTS_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f'Input folder: {INPUT_FOLDER}')
print(f'Output folder: {OUTPUT_DIR}')
print(f'Alpha for trade-off score: {ALPHA}')

Input folder: c:\Tese\results
Output folder: c:\Tese\results\benchmark_analysis_outputs
Alpha for trade-off score: 0.5


In [2]:
from IPython.display import display, Markdown

DEVICE_HINTS = {
    'pixel', 'pixel7', 'pixel8', 'pixel_7', 'pixel_8',
    'samsung', 'xiaomi', 'oneplus', 'oppo', 'vivo', 'realme',
    'huawei', 'motorola', 'moto', 'galaxy', 'redmi', 'sony', 'nokia'
}

ALGORITHM_DISPLAY_MAP = {
    'aes': 'AES',
    'aes_gcm': 'AES-GCM',
    'aesgcm': 'AES-GCM',
    'chacha20': 'ChaCha20',
    'rsa_hybrid': 'RSA-Hybrid',
    'elgamal': 'ElGamal',
    'ascon': 'ASCON',
    'elephant': 'Elephant',
    'giftcofb': 'GIFT-COFB',
    'grain128aead': 'Grain-128AEAD',
    'xoodyak': 'Xoodyak',
}


def slugify(value: str) -> str:
    value = re.sub(r'[^0-9a-zA-Z]+', '_', str(value).strip().lower())
    return re.sub(r'_+', '_', value).strip('_')


def pretty_label(value: str) -> str:
    key = slugify(value)
    if key in ALGORITHM_DISPLAY_MAP:
        return ALGORITHM_DISPLAY_MAP[key]
    return key.replace('_', ' ').title()


def normalize_filename_stem(stem: str) -> str:
    stem = re.sub(r'_per_exec$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'_bench.*$', '', stem, flags=re.IGNORECASE)
    return stem


def infer_algorithm_device_from_path(file_path: Path) -> tuple[str, str]:
    stem = normalize_filename_stem(file_path.stem)
    parts = [part for part in stem.split('_') if part]

    device_from_filename = None
    algorithm_parts = parts
    if len(parts) >= 2:
        last_part = parts[-1]
        if (
            any(hint in slugify(last_part) for hint in DEVICE_HINTS)
            or bool(re.search(r'\d', last_part))
            or last_part.lower() in DEVICE_HINTS
        ):
            device_from_filename = last_part
            algorithm_parts = parts[:-1]

    algorithm = slugify('_'.join(algorithm_parts)) if algorithm_parts else 'unknown_algorithm'
    parent_parts = [part for part in file_path.parent.parts if part not in {str(ROOT_DIR), INPUT_FOLDER.name}]
    parent_parts = [part for part in parent_parts if slugify(part) not in {slugify(ROOT_DIR.name), slugify(OUTPUT_DIR.name)}]
    folder_device = parent_parts[-1] if parent_parts else 'unknown_device'
    device = device_from_filename or folder_device or 'unknown_device'
    return algorithm, slugify(device)


def iqr_bounds(series: pd.Series) -> tuple[float, float]:
    clean = series.dropna()
    if clean.empty:
        return np.nan, np.nan
    q1 = clean.quantile(0.25)
    q3 = clean.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def add_iqr_flag(dataframe: pd.DataFrame, column: str, group_cols: list[str] | None = None) -> pd.DataFrame:
    result = dataframe.copy()
    flag_name = f'{column}_outlier'
    result[flag_name] = False

    if group_cols:
        for _, group_index in result.groupby(group_cols).groups.items():
            lower, upper = iqr_bounds(result.loc[group_index, column])
            if np.isnan(lower) or np.isnan(upper):
                continue
            result.loc[group_index, flag_name] = (
                (result.loc[group_index, column] < lower) | (result.loc[group_index, column] > upper)
            )
    else:
        lower, upper = iqr_bounds(result[column])
        if not np.isnan(lower) and not np.isnan(upper):
            result[flag_name] = (result[column] < lower) | (result[column] > upper)

    return result


def save_figure(fig: plt.Figure, filename: str) -> Path:
    output_path = PLOTS_DIR / filename
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches='tight')
    plt.close(fig)
    return output_path


def grouped_round_counts(dataframe: pd.DataFrame) -> pd.DataFrame:
    return (
        dataframe.groupby(['device', 'algorithm', 'size_bytes'])
        .agg(
            rows=('round_index', 'size'),
            unique_rounds=('round_index', 'nunique')
        )
        .reset_index()
        .sort_values(['device', 'algorithm', 'size_bytes'])
    )


def min_max_normalize(series: pd.Series) -> pd.Series:
    clean = series.astype(float)
    min_value = clean.min()
    max_value = clean.max()
    if pd.isna(min_value) or pd.isna(max_value) or np.isclose(max_value, min_value):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (clean - min_value) / (max_value - min_value)


def build_tradeoff_scores(dataframe: pd.DataFrame, alpha: float = ALPHA) -> pd.DataFrame:
    scored = dataframe.copy()
    scored['time_norm'] = min_max_normalize(scored['total_time_ms'])
    scored['energy_norm'] = min_max_normalize(scored['energy_J'])
    scored['score'] = alpha * scored['time_norm'] + (1 - alpha) * scored['energy_norm']
    return scored


def display_interpretation(text: str) -> None:
    display(Markdown(f'**Interpretation.** {text}'))


print('Helper functions ready.')

Helper functions ready.


## 1. Import Libraries and Configure Notebook

This analysis uses pandas, numpy, matplotlib, and seaborn. The notebook is configured with a consistent plotting style, display settings, and a configurable input folder and trade-off parameter.

## 2. Locate and Load All Benchmark CSV Files

The notebook searches recursively under the configured input folder, reads each CSV file, and adds a source-file column so every record remains traceable back to its origin.

In [3]:
csv_files = sorted([path for path in INPUT_FOLDER.rglob('*.csv') if path.is_file()])

if not csv_files:
    raise FileNotFoundError(f'No CSV files found under {INPUT_FOLDER}')

loaded_frames = []
load_log = []

for file_path in csv_files:
    frame = pd.read_csv(file_path)
    frame['source_file'] = file_path.name
    frame['source_path'] = str(file_path)
    loaded_frames.append(frame)
    load_log.append({'source_file': file_path.name, 'rows': len(frame)})

raw_data = pd.concat(loaded_frames, ignore_index=True)
load_summary = pd.DataFrame(load_log)

print(f'Loaded {len(csv_files)} CSV files')
print(f'Loaded {len(raw_data):,} total rows')
display(load_summary.head(20))
display(raw_data.head())

Loaded 30 CSV files
Loaded 1,780 total rows


,source_file,rows
0,aes_bench_2p10_2p20.csv,11
1,aes_bench_2p10_2p20_per_exec.csv,165
2,aesgcm_bench_2p10_2p20.csv,11
3,aesgcm_bench_2p10_2p20_per_exec.csv,165
4,chacha20_bench_2p10_2p20.csv,11
5,chacha20_bench_2p10_2p20_per_exec.csv,165
6,elgamal_bench_2p10_2p20.csv,11
7,elgamal_bench_2p10_2p20_per_exec.csv,165
8,rsa_hybrid_bench_2p10_2p20.csv,11
9,rsa_hybrid_bench_2p10_2p20_per_exec.csv,165


,size_bytes,enc_min_ns,enc_median_ns,enc_max_ns,enc_mean_ns,enc_std_ns,enc_std_percent,dec_min_ns,dec_median_ns,dec_max_ns,dec_mean_ns,dec_std_ns,dec_std_percent,energy_mWh,source_file,source_path,round_index,enc_ns,dec_ns,enc_mem_bytes,dec_mem_bytes,method
0,1024,"316,700.0000","639,000.0000","6,459,200.0000","1,434,700.0000","1,757,402.0000",122.4900,"677,200.0000","784,300.0000","2,622,900.0000","1,146,993.0000","582,203.0000",50.7600,0.1280,aes_bench_2p10_2p20.csv,c:\Tese\results\time\aes_bench_2p10_2p20.csv,NaN,NaN,NaN,NaN,NaN,NaN
1,2048,"597,100.0000","718,500.0000","1,353,000.0000","767,800.0000","191,843.0000",24.9900,"426,200.0000","1,376,000.0000","2,359,000.0000","1,302,580.0000","611,523.0000",46.9500,0.1270,aes_bench_2p10_2p20.csv,c:\Tese\results\time\aes_bench_2p10_2p20.csv,NaN,NaN,NaN,NaN,NaN,NaN
2,4096,"424,400.0000","535,900.0000","2,638,500.0000","851,613.0000","685,626.0000",80.5100,"365,900.0000","487,000.0000","1,814,600.0000","628,086.0000","437,749.0000",69.7000,0.1287,aes_bench_2p10_2p20.csv,c:\Tese\results\time\aes_bench_2p10_2p20.csv,NaN,NaN,NaN,NaN,NaN,NaN
3,8192,"459,000.0000","944,100.0000","4,226,500.0000","1,269,346.0000","986,485.0000",77.7200,"565,200.0000","944,900.0000","1,538,500.0000","919,353.0000","277,405.0000",30.1700,0.1274,aes_bench_2p10_2p20.csv,c:\Tese\results\time\aes_bench_2p10_2p20.csv,NaN,NaN,NaN,NaN,NaN,NaN
4,16384,"603,000.0000","805,600.0000","3,672,300.0000","1,056,973.0000","766,369.0000",72.5100,"844,300.0000","1,219,400.0000","2,254,800.0000","1,350,273.0000","470,314.0000",34.8300,0.1287,aes_bench_2p10_2p20.csv,c:\Tese\results\time\aes_bench_2p10_2p20.csv,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Extract Algorithm and Device Metadata from Filenames

The notebook standardizes algorithm and device labels so results can be grouped consistently across folders and filename variants.

In [4]:
schema_reports = []
file_meta_rows = []

for file_path in csv_files:
    try:
        preview = pd.read_csv(file_path, nrows=5)
        columns = list(preview.columns)
        is_expected_schema = all(column in columns for column in EXPECTED_COLUMNS)
        algorithm, device = infer_algorithm_device_from_path(file_path)
        file_meta_rows.append({
            'source_file': file_path.name,
            'source_path': str(file_path),
            'algorithm': algorithm,
            'device': device,
            'schema_type': 'raw_expected' if is_expected_schema else 'other',
            'columns': ', '.join(columns),
        })
    except Exception as exc:
        schema_reports.append({'source_file': file_path.name, 'error': str(exc)})

file_metadata = pd.DataFrame(file_meta_rows)
if not schema_reports:
    print('Schema inspection completed successfully.')
else:
    print('Some files could not be inspected.')

display(file_metadata.groupby(['schema_type', 'device'], dropna=False).size().reset_index(name='file_count').sort_values(['schema_type', 'device']))
display(file_metadata.head(20))

Schema inspection completed successfully.


,schema_type,device,file_count
0,other,time,10
1,raw_expected,d1,10
2,raw_expected,d2,10


,source_file,source_path,algorithm,device,schema_type,columns
0,aes_bench_2p10_2p20.csv,c:\Tese\results\time\aes_bench_2p10_2p20.csv,aes,time,other,"size_bytes, enc_min_ns, enc_median_ns, enc_max..."
1,aes_bench_2p10_2p20_per_exec.csv,c:\Tese\results\time\aes_bench_2p10_2p20_per_e...,aes,time,other,"size_bytes, round_index, enc_ns, dec_ns, energ..."
2,aesgcm_bench_2p10_2p20.csv,c:\Tese\results\time\aesgcm_bench_2p10_2p20.csv,aesgcm,time,other,"size_bytes, enc_min_ns, enc_median_ns, enc_max..."
3,aesgcm_bench_2p10_2p20_per_exec.csv,c:\Tese\results\time\aesgcm_bench_2p10_2p20_pe...,aesgcm,time,other,"size_bytes, round_index, enc_ns, dec_ns, energ..."
4,chacha20_bench_2p10_2p20.csv,c:\Tese\results\time\chacha20_bench_2p10_2p20.csv,chacha20,time,other,"size_bytes, enc_min_ns, enc_median_ns, enc_max..."
5,chacha20_bench_2p10_2p20_per_exec.csv,c:\Tese\results\time\chacha20_bench_2p10_2p20_...,chacha20,time,other,"size_bytes, round_index, enc_ns, dec_ns, energ..."
6,elgamal_bench_2p10_2p20.csv,c:\Tese\results\time\elgamal_bench_2p10_2p20.csv,elgamal,time,other,"size_bytes, enc_min_ns, enc_median_ns, enc_max..."
7,elgamal_bench_2p10_2p20_per_exec.csv,c:\Tese\results\time\elgamal_bench_2p10_2p20_p...,elgamal,time,other,"size_bytes, round_index, enc_ns, dec_ns, energ..."
8,rsa_hybrid_bench_2p10_2p20.csv,c:\Tese\results\time\rsa_hybrid_bench_2p10_2p2...,rsa_hybrid,time,other,"size_bytes, enc_min_ns, enc_median_ns, enc_max..."
9,rsa_hybrid_bench_2p10_2p20_per_exec.csv,c:\Tese\results\time\rsa_hybrid_bench_2p10_2p2...,rsa_hybrid,time,other,"size_bytes, round_index, enc_ns, dec_ns, energ..."


## 4. Convert Units and Add Derived Metrics

The raw benchmark measurements are converted to milliseconds, Joules, kilobytes, and megabytes. Derived metrics such as total time and total memory usage are added to support ranking and visualization.

In [5]:
raw_files = file_metadata.loc[file_metadata['schema_type'] == 'raw_expected'].copy()
other_files = file_metadata.loc[file_metadata['schema_type'] != 'raw_expected'].copy()

benchmark_frames = []
skipped_files = []

for _, row in raw_files.iterrows():
    file_path = Path(row['source_path'])
    try:
        frame = pd.read_csv(file_path, usecols=EXPECTED_COLUMNS)
        frame['source_file'] = row['source_file']
        frame['source_path'] = row['source_path']
        frame['algorithm'] = row['algorithm']
        frame['device'] = row['device']
        benchmark_frames.append(frame)
    except Exception as exc:
        skipped_files.append({'source_file': row['source_file'], 'reason': str(exc)})

if not benchmark_frames:
    raise RuntimeError('No raw benchmark CSV files could be loaded.')

benchmark_data = pd.concat(benchmark_frames, ignore_index=True)
benchmark_data = benchmark_data.rename(columns={'method': 'method_raw'})

for column in ['size_bytes', 'round_index', 'enc_ns', 'dec_ns', 'enc_mem_bytes', 'dec_mem_bytes', 'energy_mWh']:
    benchmark_data[column] = pd.to_numeric(benchmark_data[column], errors='coerce')

benchmark_data['algorithm'] = benchmark_data['algorithm'].map(pretty_label)
benchmark_data['device'] = benchmark_data['device'].map(pretty_label)
benchmark_data['method'] = benchmark_data['method_raw'].astype(str).str.strip().str.upper()

benchmark_data['enc_ms'] = benchmark_data['enc_ns'] / 1_000_000
benchmark_data['dec_ms'] = benchmark_data['dec_ns'] / 1_000_000
benchmark_data['total_time_ms'] = benchmark_data['enc_ms'] + benchmark_data['dec_ms']
benchmark_data['energy_J'] = benchmark_data['energy_mWh'] * 3.6
benchmark_data['size_kb'] = benchmark_data['size_bytes'] / 1024
benchmark_data['size_mb'] = benchmark_data['size_bytes'] / (1024 ** 2)
benchmark_data['avg_mem_bytes'] = benchmark_data[['enc_mem_bytes', 'dec_mem_bytes']].mean(axis=1)
benchmark_data['total_mem_bytes'] = benchmark_data['enc_mem_bytes'] + benchmark_data['dec_mem_bytes']
benchmark_data['throughput_kb_per_ms'] = benchmark_data['size_kb'] / benchmark_data['total_time_ms'].replace(0, np.nan)

print(f'Raw benchmark rows loaded: {len(benchmark_data):,}')
print(f'Skipped non-raw files: {len(other_files):,}')
if skipped_files:
    display(pd.DataFrame(skipped_files))

display(benchmark_data.head())

Raw benchmark rows loaded: 900
Skipped non-raw files: 10


,size_bytes,round_index,enc_ns,dec_ns,enc_mem_bytes,dec_mem_bytes,energy_mWh,method_raw,source_file,source_path,algorithm,device,method,enc_ms,dec_ms,total_time_ms,energy_J,size_kb,size_mb,avg_mem_bytes,total_mem_bytes,throughput_kb_per_ms
0,1024,1,1292332,1416800,11796,6717,0.0000,INTEGRATION,aes_bench_2p10_2p18.csv,c:\Tese\results\unplugged_run\d1\aes_bench_2p1...,AES,D1,INTEGRATION,1.2923,1.4168,2.7091,0.0001,1.0000,0.0010,"9,256.5000",18513,0.3691
1,1024,2,1399292,1571836,11632,6881,0.0000,INTEGRATION,aes_bench_2p10_2p18.csv,c:\Tese\results\unplugged_run\d1\aes_bench_2p1...,AES,D1,INTEGRATION,1.3993,1.5718,2.9711,0.0001,1.0000,0.0010,"9,256.5000",18513,0.3366
2,1024,3,1409396,1556608,11632,6881,0.0000,INTEGRATION,aes_bench_2p10_2p18.csv,c:\Tese\results\unplugged_run\d1\aes_bench_2p1...,AES,D1,INTEGRATION,1.4094,1.5566,2.9660,0.0001,1.0000,0.0010,"9,256.5000",18513,0.3372
3,1024,4,1473850,1536726,11632,6717,0.0000,INTEGRATION,aes_bench_2p10_2p18.csv,c:\Tese\results\unplugged_run\d1\aes_bench_2p1...,AES,D1,INTEGRATION,1.4739,1.5367,3.0106,0.0001,1.0000,0.0010,"9,174.5000",18349,0.3322
4,1024,5,1432411,1560802,11796,6717,0.0000,INTEGRATION,aes_bench_2p10_2p18.csv,c:\Tese\results\unplugged_run\d1\aes_bench_2p1...,AES,D1,INTEGRATION,1.4324,1.5608,2.9932,0.0001,1.0000,0.0010,"9,256.5000",18513,0.3341


## 5. Validate Data Quality

This step checks schema consistency, missing values, duplicated rows, impossible numeric values, expected round counts, and IQR-based outliers without removing the original observations.

In [6]:
missing_summary = benchmark_data.isna().sum().sort_values(ascending=False)
duplicate_rows = benchmark_data.duplicated().sum()
invalid_numeric_mask = (
    (benchmark_data['size_bytes'] <= 0)
    | (benchmark_data['round_index'] <= 0)
    | (benchmark_data['enc_ns'] < 0)
    | (benchmark_data['dec_ns'] < 0)
    | (benchmark_data['enc_mem_bytes'] < 0)
    | (benchmark_data['dec_mem_bytes'] < 0)
    | (benchmark_data['energy_mWh'] < 0)
)
invalid_numeric_rows = benchmark_data.loc[invalid_numeric_mask].copy()

benchmark_data = benchmark_data.copy()
benchmark_data = add_iqr_flag(benchmark_data, 'enc_ms', ['algorithm', 'device', 'size_bytes'])
benchmark_data = add_iqr_flag(benchmark_data, 'dec_ms', ['algorithm', 'device', 'size_bytes'])
benchmark_data = add_iqr_flag(benchmark_data, 'total_time_ms', ['algorithm', 'device', 'size_bytes'])
benchmark_data = add_iqr_flag(benchmark_data, 'energy_J', ['algorithm', 'device', 'size_bytes'])
benchmark_data = add_iqr_flag(benchmark_data, 'enc_mem_bytes', ['algorithm', 'device', 'size_bytes'])
benchmark_data = add_iqr_flag(benchmark_data, 'dec_mem_bytes', ['algorithm', 'device', 'size_bytes'])

outlier_columns = [column for column in benchmark_data.columns if column.endswith('_outlier')]
benchmark_data['any_outlier'] = benchmark_data[outlier_columns].any(axis=1)

round_counts = grouped_round_counts(benchmark_data)
expected_rounds = round_counts['unique_rounds'].mode().iloc[0] if not round_counts.empty else np.nan
inconsistent_round_counts = round_counts.loc[round_counts['unique_rounds'] != expected_rounds].copy() if not np.isnan(expected_rounds) else round_counts.iloc[0:0].copy()

print('Missing values per column:')
display(missing_summary.to_frame(name='missing_count'))
print(f'Duplicated rows: {duplicate_rows}')
print(f'Rows with invalid numeric values: {len(invalid_numeric_rows)}')
print(f"Detected outlier rows: {int(benchmark_data['any_outlier'].sum())}")
print(f'Expected rounds per size/algorithm/device combination: {expected_rounds}')
if not invalid_numeric_rows.empty:
    display(invalid_numeric_rows.head(20))
if not inconsistent_round_counts.empty:
    display(inconsistent_round_counts.head(20))

display(benchmark_data[['algorithm', 'device', 'size_bytes', 'round_index', 'any_outlier']].head())

Missing values per column:


,missing_count
size_bytes,0
round_index,0
enc_ns,0
dec_ns,0
enc_mem_bytes,0
dec_mem_bytes,0
energy_mWh,0
method_raw,0
source_file,0
source_path,0


Duplicated rows: 0
Rows with invalid numeric values: 0
Detected outlier rows: 300
Expected rounds per size/algorithm/device combination: 5


,algorithm,device,size_bytes,round_index,any_outlier
0,AES,D1,1024,1,True
1,AES,D1,1024,2,False
2,AES,D1,1024,3,False
3,AES,D1,1024,4,False
4,AES,D1,1024,5,False


## 6. Aggregate Results by Algorithm, Device, and Input Size

Grouped summaries provide the basis for comparisons across algorithms, devices, and file sizes. The aggregated table is exported for downstream reporting and thesis figures.

In [ ]:
metric_columns = [
    'enc_ms', 'dec_ms', 'total_time_ms', 'energy_J',
    'enc_mem_bytes', 'dec_mem_bytes', 'avg_mem_bytes', 'total_mem_bytes',
    'throughput_kb_per_ms'
]

aggregated_results = (
    benchmark_data
    .groupby(['algorithm', 'device', 'size_bytes'], as_index=False)
    .agg(
        rounds=('round_index', 'count'),
        enc_ms_mean=('enc_ms', 'mean'),
        enc_ms_median=('enc_ms', 'median'),
        enc_ms_std=('enc_ms', 'std'),
        enc_ms_min=('enc_ms', 'min'),
        enc_ms_max=('enc_ms', 'max'),
        dec_ms_mean=('dec_ms', 'mean'),
        dec_ms_median=('dec_ms', 'median'),
        dec_ms_std=('dec_ms', 'std'),
        dec_ms_min=('dec_ms', 'min'),
        dec_ms_max=('dec_ms', 'max'),
        total_time_ms_mean=('total_time_ms', 'mean'),
        total_time_ms_median=('total_time_ms', 'median'),
        total_time_ms_std=('total_time_ms', 'std'),
        total_time_ms_min=('total_time_ms', 'min'),
        total_time_ms_max=('total_time_ms', 'max'),
        energy_J_mean=('energy_J', 'mean'),
        energy_J_median=('energy_J', 'median'),
        energy_J_std=('energy_J', 'std'),
        energy_J_min=('energy_J', 'min'),
        energy_J_max=('energy_J', 'max'),
        enc_mem_bytes_mean=('enc_mem_bytes', 'mean'),
        enc_mem_bytes_median=('enc_mem_bytes', 'median'),
        enc_mem_bytes_std=('enc_mem_bytes', 'std'),
        enc_mem_bytes_min=('enc_mem_bytes', 'min'),
        enc_mem_bytes_max=('enc_mem_bytes', 'max'),
        dec_mem_bytes_mean=('dec_mem_bytes', 'mean'),
        dec_mem_bytes_median=('dec_mem_bytes', 'median'),
        dec_mem_bytes_std=('dec_mem_bytes', 'std'),
        dec_mem_bytes_min=('dec_mem_bytes', 'min'),
        dec_mem_bytes_max=('dec_mem_bytes', 'max'),
        total_mem_bytes_mean=('total_mem_bytes', 'mean'),
        total_mem_bytes_median=('total_mem_bytes', 'median'),
        total_mem_bytes_std=('total_mem_bytes', 'std'),
        total_mem_bytes_min=('total_mem_bytes', 'min'),
        total_mem_bytes_max=('total_mem_bytes', 'max'),
        throughput_kb_per_ms_mean=('throughput_kb_per_ms', 'mean'),
        throughput_kb_per_ms_median=('throughput_kb_per_ms', 'median'),
        throughput_kb_per_ms_std=('throughput_kb_per_ms', 'std'),
        throughput_kb_per_ms_min=('throughput_kb_per_ms', 'min'),
        throughput_kb_per_ms_max=('throughput_kb_per_ms', 'max'),
    )
    .sort_values(['device', 'algorithm', 'size_bytes'])
    .reset_index(drop=True)
)

aggregated_results['size_kb'] = aggregated_results['size_bytes'] / 1024
aggregated_results['size_mb'] = aggregated_results['size_bytes'] / (1024 ** 2)

# Add coefficient of variation (CV) in percentages for key metrics
# CV (%) = (std / mean) * 100, provides a standardized measure of variability
aggregated_results['total_time_cv_pct'] = (aggregated_results['total_time_ms_std'] / aggregated_results['total_time_ms_mean']) * 100
aggregated_results['energy_cv_pct'] = (aggregated_results['energy_J_std'] / aggregated_results['energy_J_mean']) * 100
aggregated_results['enc_ms_cv_pct'] = (aggregated_results['enc_ms_std'] / aggregated_results['enc_ms_mean']) * 100
aggregated_results['dec_ms_cv_pct'] = (aggregated_results['dec_ms_std'] / aggregated_results['dec_ms_mean']) * 100
aggregated_results['total_mem_cv_pct'] = (aggregated_results['total_mem_bytes_std'] / aggregated_results['total_mem_bytes_mean']) * 100

aggregated_path = TABLES_DIR / 'aggregated_results.csv'
aggregated_results.to_csv(aggregated_path, index=False)

print(f'Aggregated rows: {len(aggregated_results):,}')
print(f'Saved aggregated results to: {aggregated_path}')
print('\nCoefficiente de Variação (CV) em percentagens foi adicionado.')
print('CV (%) = (Desvio Padrão / Média) × 100 - Mede a variabilidade relativa.')
display(aggregated_results[['algorithm', 'device', 'size_bytes', 'total_time_ms_mean', 'total_time_cv_pct', 'energy_J_mean', 'energy_cv_pct']].head(20))

Aggregated rows: 180
Saved aggregated results to: c:\Tese\results\benchmark_analysis_outputs\tables\aggregated_results.csv


,algorithm,device,size_bytes,rounds,enc_ms_mean,enc_ms_median,enc_ms_std,enc_ms_min,enc_ms_max,dec_ms_mean,dec_ms_median,dec_ms_std,dec_ms_min,dec_ms_max,total_time_ms_mean,total_time_ms_median,total_time_ms_std,total_time_ms_min,total_time_ms_max,energy_J_mean,energy_J_median,energy_J_std,energy_J_min,energy_J_max,enc_mem_bytes_mean,enc_mem_bytes_median,enc_mem_bytes_std,enc_mem_bytes_min,enc_mem_bytes_max,dec_mem_bytes_mean,dec_mem_bytes_median,dec_mem_bytes_std,dec_mem_bytes_min,dec_mem_bytes_max,total_mem_bytes_mean,total_mem_bytes_median,total_mem_bytes_std,total_mem_bytes_min,total_mem_bytes_max,throughput_kb_per_ms_mean,throughput_kb_per_ms_median,throughput_kb_per_ms_std,throughput_kb_per_ms_min,throughput_kb_per_ms_max,size_kb,size_mb
0,AES,D1,1024,5,1.4015,1.4094,0.0674,1.2923,1.4739,1.5286,1.5566,0.0637,1.4168,1.5718,2.9300,2.9711,0.1248,2.7091,3.0106,0.0001,0.0001,0.0000,0.0001,0.0001,"11,697.6000","11,632.0000",89.8265,11632,11796,"6,782.6000","6,717.0000",89.8265,6717,6881,"18,480.2000","18,513.0000",73.3430,18349,18513,0.3418,0.3366,0.0154,0.3322,0.3691,1.0000,0.0010
1,AES,D1,2048,5,1.4900,1.4635,0.0711,1.4200,1.5997,1.6473,1.6693,0.0563,1.5619,1.7058,3.1373,3.1258,0.0973,3.0254,3.2690,0.0001,0.0001,0.0000,0.0001,0.0001,"21,822.8000","21,790.0000",73.3430,21790,21954,"11,927.2000","11,960.0000",73.3430,11796,11960,"33,750.0000","33,750.0000",115.9655,33586,33914,0.6380,0.6398,0.0197,0.6118,0.6611,2.0000,0.0020
2,AES,D1,4096,5,1.5367,1.4919,0.1259,1.4359,1.7526,1.7391,1.7179,0.0988,1.6564,1.9023,3.2758,3.1604,0.2205,3.1255,3.6549,0.0002,0.0002,0.0000,0.0002,0.0002,"42,008.2000","41,943.0000",89.2788,41943,42106,"22,446.0000","22,446.0000",0.0000,22446,22446,"64,454.2000","64,389.0000",89.2788,64389,64552,1.2252,1.2656,0.0769,1.0944,1.2798,4.0000,0.0039
3,AES,D1,8192,5,1.8349,1.7707,0.1419,1.7165,2.0715,2.3948,2.3515,0.1220,2.2868,2.5866,4.2297,4.1975,0.1773,4.0575,4.4441,0.0002,0.0002,0.0000,0.0002,0.0002,"81,509.6000","85,032.0000","8,152.4964",66928,85360,"43,909.0000","43,909.0000",0.0000,43909,43909,"125,418.6000","128,941.0000","8,152.4964",110837,129269,1.8940,1.9059,0.0789,1.8001,1.9716,8.0000,0.0078
4,AES,D1,16384,5,2.2509,2.2008,0.1126,2.1580,2.4248,3.0425,3.0840,0.1542,2.8710,3.2487,5.2934,5.3329,0.1506,5.0398,5.4066,0.0004,0.0004,0.0000,0.0004,0.0004,"119,205.2000","118,549.0000","1,575.9499",117567,121335,"41,690.4000","42,282.0000","1,367.9511",39415,42815,"160,895.6000","161,037.0000","2,185.5390",157964,163823,3.0247,3.0002,0.0886,2.9593,3.1747,16.0000,0.0156
5,AES,D1,32768,5,3.3390,3.3263,0.0312,3.3121,3.3796,4.4368,4.4061,0.2127,4.2551,4.7749,7.7758,7.7707,0.2357,7.5672,8.1545,0.0007,0.0007,0.0000,0.0007,0.0007,"97,672.6000","94,027.0000","56,169.1428",39058,157133,"40,734.6000","39,414.0000","5,917.4317",36103,51007,"138,407.2000","145,034.0000","54,951.4654",78472,194704,4.1183,4.1180,0.1222,3.9242,4.2288,32.0000,0.0312
6,AES,D1,65536,5,5.3889,5.3808,0.0428,5.3386,5.4459,7.4862,7.4006,0.2620,7.3072,7.9450,12.8751,12.7820,0.2599,12.6695,13.3258,0.0013,0.0013,0.0000,0.0013,0.0013,"90,737.8000","77,601.0000","42,309.8288",51388,136748,"49,433.4000","49,081.0000","8,096.4881",38472,60222,"140,171.2000","130,861.0000","48,382.7582",91818,196970,4.9724,5.0071,0.0981,4.8027,5.0515,64.0000,0.0625
7,AES,D1,131072,5,9.7849,9.7437,0.0965,9.7196,9.9546,13.6304,13.5960,0.1425,13.5050,13.8750,23.4153,23.3653,0.1415,23.2423,23.5946,0.0025,0.0025,0.0000,0.0025,0.0025,"117,886.6000","113,604.0000","19,868.8364",100743,147876,"72,455.4000","69,110.0000","27,726.8295",50188,119627,"190,342.0000","195,576.0000","23,963.8724",154985,220371,5.4667,5.4782,0.0330,5.4250,5.5072,128.0000,0.1250
8,AES,D1,262144,5,18.2335,18.1891,0.3318,17.8333,18.6249,26.2988,26.4365,0.2771,25.9848,26.5443,44.5324,44.3496,0.4927,44.0240,45.0614,0.0049,0.0049,0.0000,0.0049,0.0049,"169,483.8000","191,621.0000","78,518.2629",63663,252297,"60,791.6000","61,259.0000","24,184.6650",34634,96197,"230,275.4000","252,880.0000","73,777.5429",149263

## 7. Create Comparative Plots

The following figures compare performance across input sizes, algorithms, and devices. Each figure is saved to the plots folder and followed by a short interpretation suitable for a thesis or report.

In [14]:
# Encryption, decryption, and total time trends.
fig, axes = plt.subplots(1, 3, figsize=(21, 6), sharex=True)

sns.lineplot(
    data=plot_source,
    x='size_bytes',
    y='enc_ms_mean',
    hue='algorithm',
    style='device',
    markers=True,
    dashes=True,
    ax=axes[0],
)
sns.lineplot(
    data=plot_source,
    x='size_bytes',
    y='dec_ms_mean',
    hue='algorithm',
    style='device',
    markers=True,
    dashes=True,
    ax=axes[1],
    legend=False,
)
sns.lineplot(
    data=plot_source,
    x='size_bytes',
    y='total_time_ms_mean',
    hue='algorithm',
    style='device',
    markers=True,
    dashes=True,
    ax=axes[2],
    legend=False,
)

axes[0].set_title('Encryption Time vs Input Size')
axes[1].set_title('Decryption Time vs Input Size')
axes[2].set_title('Total Time vs Input Size')
for axis in axes:
    axis.set_xscale('log', base=2)
    axis.set_xlabel('Input size (bytes, log2 scale)')
    axis.set_ylabel('Time (ms)')

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    axes[0].legend(handles, labels, title='Algorithm / Device', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    axes[0].get_legend().set_title('Algorithm / Device')
    axes[0].get_legend().set_bbox_to_anchor((1.02, 1))
    axes[0].get_legend()._loc = 2
for axis in axes[1:]:
    legend = axis.get_legend()
    if legend is not None:
        legend.remove()

fig.suptitle('Benchmark Time Trends by Algorithm and Device', y=1.02)
fig.tight_layout()
save_figure(fig, 'time_trends_by_algorithm_device.png')

display_interpretation('Encryption and decryption times generally increase with input size, while the total-time plot gives the clearest overall scaling picture. The legend distinguishes both algorithm and device, which makes it easier to see whether a method is fast because of the implementation itself or because it benefits from a specific device.')

# Energy and memory trends.
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

sns.lineplot(
    data=plot_source,
    x='size_bytes',
    y='energy_J_mean',
    hue='algorithm',
    style='device',
    markers=True,
    dashes=True,
    ax=axes[0],
)
sns.lineplot(
    data=plot_source,
    x='size_bytes',
    y='total_mem_bytes_mean',
    hue='algorithm',
    style='device',
    markers=True,
    dashes=True,
    ax=axes[1],
    legend=False,
)

axes[0].set_title('Energy Consumption vs Input Size')
axes[1].set_title('Memory Usage vs Input Size')
for axis in axes:
    axis.set_xscale('log', base=2)
    axis.set_xlabel('Input size (bytes, log2 scale)')
axes[0].set_ylabel('Energy (J)')
axes[1].set_ylabel('Memory (bytes)')
legend = axes[0].get_legend()
if legend is not None:
    legend.set_title('Algorithm / Device')
    legend.set_bbox_to_anchor((1.02, 1))
    legend._loc = 2
axes[1].get_legend().remove() if axes[1].get_legend() is not None else None
fig.tight_layout()
save_figure(fig, 'energy_memory_trends.png')

display_interpretation('Energy and memory grow with input size, but the slopes differ by algorithm and device. Lower and smoother curves indicate more resource-efficient implementations, which is important for mobile battery life and thermal stability.')

**Interpretation.** Encryption and decryption times generally increase with input size, while the total-time plot gives the clearest overall scaling picture. The legend distinguishes both algorithm and device, which makes it easier to see whether a method is fast because of the implementation itself or because it benefits from a specific device.

**Interpretation.** Energy and memory grow with input size, but the slopes differ by algorithm and device. Lower and smoother curves indicate more resource-efficient implementations, which is important for mobile battery life and thermal stability.

In [9]:
# Comparison of algorithms for each device.
algorithm_device_plot = sns.relplot(
    data=plot_source,
    x='size_bytes',
    y='total_time_ms_mean',
    hue='algorithm',
    col='device',
    kind='line',
    marker='o',
    facet_kws={'sharey': False, 'sharex': True},
    height=5.5,
    aspect=1.15,
)
algorithm_device_plot.set(xscale='log')
algorithm_device_plot.set_axis_labels('Input size (bytes, log scale)', 'Total time (ms)')
algorithm_device_plot.set_titles('Device: {col_name}')
algorithm_device_plot.figure.suptitle('Algorithm Comparison Within Each Device', y=1.03)
algorithm_device_plot.figure.tight_layout()
algorithm_device_plot.figure.savefig(PLOTS_DIR / 'algorithm_comparison_by_device.png', bbox_inches='tight', dpi=300)
plt.close(algorithm_device_plot.figure)

display_interpretation('This comparison highlights whether an algorithm remains competitive on both devices or only on one platform. Algorithms with small cross-device gaps are easier to justify for deployment because they are less sensitive to device-specific runtime behavior.')

# Comparison of devices for each algorithm.
device_algorithm_plot = sns.relplot(
    data=plot_source,
    x='size_bytes',
    y='total_time_ms_mean',
    hue='device',
    col='algorithm',
    col_wrap=3,
    kind='line',
    marker='o',
    facet_kws={'sharey': False, 'sharex': True},
    height=4.4,
    aspect=1.15,
)
device_algorithm_plot.set(xscale='log')
device_algorithm_plot.set_axis_labels('Input size (bytes, log scale)', 'Total time (ms)')
device_algorithm_plot.set_titles('{col_name}')
device_algorithm_plot.figure.suptitle('Device Comparison Within Each Algorithm', y=1.02)
device_algorithm_plot.figure.tight_layout()
device_algorithm_plot.figure.savefig(PLOTS_DIR / 'device_comparison_by_algorithm.png', bbox_inches='tight', dpi=300)
plt.close(device_algorithm_plot.figure)

display_interpretation('This figure shows how much the hardware choice changes the benchmark outcome for each algorithm. A smaller device gap indicates more portable performance, while a larger gap suggests the implementation benefits strongly from the faster device.')

**Interpretation.** This comparison highlights whether an algorithm remains competitive on both devices or only on one platform. Algorithms with small cross-device gaps are easier to justify for deployment because they are less sensitive to device-specific runtime behavior.

**Interpretation.** This figure shows how much the hardware choice changes the benchmark outcome for each algorithm. A smaller device gap indicates more portable performance, while a larger gap suggests the implementation benefits strongly from the faster device.

### Per-Device Figures

The following figures create one PNG per device so the final report can include device-specific performance summaries.

## 8. Analyze Round-Level Variability

Boxplots show how stable each algorithm is across repeated rounds. Wider boxes or longer whiskers indicate greater variability and a less predictable runtime profile.

In [10]:
variability_plot = sns.catplot(
    data=benchmark_data,
    x='size_bytes',
    y='total_time_ms',
    hue='device',
    col='algorithm',
    col_wrap=3,
    kind='box',
    height=4.2,
    aspect=1.15,
    sharey=False,
)
variability_plot.set(xscale='log')
variability_plot.set_axis_labels('Input size (bytes, log scale)', 'Total time (ms)')
variability_plot.set_titles('{col_name}')
variability_plot.figure.suptitle('Round-Level Variability in Total Time', y=1.02)
variability_plot.figure.tight_layout()
variability_plot.figure.savefig(PLOTS_DIR / 'round_level_variability_total_time.png', bbox_inches='tight', dpi=300)
plt.close(variability_plot.figure)

display_interpretation('The boxplots reveal whether a method behaves consistently across repeated rounds or shows occasional spikes. Tight distributions indicate stable performance, which is valuable in benchmark reporting because it strengthens the reliability of the average values.')

**Interpretation.** The boxplots reveal whether a method behaves consistently across repeated rounds or shows occasional spikes. Tight distributions indicate stable performance, which is valuable in benchmark reporting because it strengthens the reliability of the average values.

## 9. Build Summary Ranking Tables

The summary tables identify the leading algorithm under different criteria, including per-device speed, per-device energy use, and best performance at each input size.

## 8.5 Statistical Analysis: Coefficient of Variation and Stability Visualization

This section creates additional statistical visualizations to better understand data variability and stability across algorithms, including histograms, distribution plots, and coefficient of variation heatmaps.

In [ ]:
# Statistical Analysis: Coefficient of Variation and Distribution Visualization
# These plots provide deeper statistical insights into algorithm stability and variability

# 1. Coefficient of Variation (CV) by Algorithm
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cv_data = aggregated_results.groupby('algorithm')[['total_time_cv_pct', 'energy_cv_pct']].mean().sort_values('total_time_cv_pct', ascending=False).reset_index()

sns.barplot(data=cv_data, x='algorithm', y='total_time_cv_pct', ax=axes[0], color='#4c72b0', palette='muted')
axes[0].set_title('Coefficient of Variation: Total Time (%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Algorithm')
axes[0].set_ylabel('CV (%)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

sns.barplot(data=cv_data, x='algorithm', y='energy_cv_pct', ax=axes[1], color='#dd8452', palette='muted')
axes[1].set_title('Coefficient of Variation: Energy (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Algorithm')
axes[1].set_ylabel('CV (%)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('Algorithm Stability: Lower CV (%) = More Consistent Performance', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
save_figure(fig, 'coefficient_of_variation_analysis.png')

display_interpretation(
    'Coefficient of Variation (CV) normalizes standard deviation relative to the mean, making it a dimensionless measure of stability. '
    'Lower CV indicates more consistent and predictable performance. Algorithms with CV < 10% exhibit excellent stability, '
    'while CV > 20% suggests higher variability that could affect real-time responsiveness.'
)

# 2. Histograms of Total Time Distribution by Algorithm
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
algorithms = sorted(benchmark_data['algorithm'].unique())[:6]

for idx, algo in enumerate(algorithms):
    data_subset = benchmark_data[benchmark_data['algorithm'] == algo]['total_time_ms']
    axes[idx].hist(data_subset, bins=30, color='#4c72b0', alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{algo}', fontweight='bold')
    axes[idx].set_xlabel('Total Time (ms)')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add statistics to the plot
    mean_val = data_subset.mean()
    std_val = data_subset.std()
    axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}ms')
    axes[idx].axvline(mean_val + std_val, color='orange', linestyle='--', linewidth=1.5, alpha=0.7, label=f'±1 Std')
    axes[idx].axvline(mean_val - std_val, color='orange', linestyle='--', linewidth=1.5, alpha=0.7)
    axes[idx].legend(fontsize=8)

fig.suptitle('Distribution of Total Execution Times by Algorithm', fontsize=14, fontweight='bold', y=1.00)
fig.tight_layout()
save_figure(fig, 'time_distribution_histograms.png')

display_interpretation(
    'These histograms visualize the shape of the time distribution for each algorithm. A tight, bell-shaped curve indicates '
    'normal and predictable performance. Skewed or multimodal distributions may indicate algorithm-specific behavior variations '
    'or device-dependent performance changes. The red line marks the mean, and orange lines show ±1 standard deviation.'
)

# 3. Violin Plots: Distribution shapes by Device and Algorithm
fig = plt.figure(figsize=(16, 8))
sns.violinplot(
    data=benchmark_data,
    x='algorithm',
    y='total_time_ms',
    hue='device',
    split=False,
    inner='box',
)
plt.title('Distribution Shape of Total Time by Algorithm and Device', fontsize=14, fontweight='bold')
plt.xlabel('Algorithm')
plt.ylabel('Total Time (ms)')
plt.yscale('log')
plt.xticks(rotation=45)
plt.tight_layout()
save_figure(fig, 'time_distribution_violin_plots.png')

display_interpretation(
    'Violin plots combine boxplot and density information to show the full distribution shape. '
    'Wider sections indicate higher data density. This visualization reveals whether algorithms behave differently '
    'across devices and helps identify multimodal behavior or long tails (outliers) in the distribution.'
)

# 4. Scatterplot: Energy vs Time Trade-off with Algorithm and Device color coding
fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(
    benchmark_data['total_time_ms'],
    benchmark_data['energy_J'],
    c=pd.factorize(benchmark_data['algorithm'])[0],
    s=100,
    alpha=0.6,
    cmap='tab20',
    edgecolors='black',
    linewidth=0.5
)
ax.set_xlabel('Total Time (ms)', fontsize=12)
ax.set_ylabel('Energy (J)', fontsize=12)
ax.set_title('Trade-off Space: Energy vs Time', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Create legend with algorithm names
algorithms_unique = sorted(benchmark_data['algorithm'].unique())
cbar = plt.colorbar(scatter, ax=ax, ticks=range(len(algorithms_unique)))
cbar.ax.set_yticklabels(algorithms_unique)
cbar.set_label('Algorithm')

plt.tight_layout()
save_figure(fig, 'energy_time_tradeoff_scatter.png')

display_interpretation(
    'This scatter plot visualizes the fundamental trade-off between execution time and energy consumption. '
    'Algorithms clustered in the lower-left corner are both fast and energy-efficient (optimal), while those in the '
    'upper-right are slow and power-hungry (suboptimal). Points along a diagonal suggest algorithms that trade one metric for the other.'
)

# 5. Heatmap: Coefficient of Variation by Algorithm and Device
cv_heatmap_data = aggregated_results.pivot_table(
    values='total_time_cv_pct',
    index='algorithm',
    columns='device',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cv_heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn_r', cbar_kws={'label': 'CV (%)'}, ax=ax, linewidths=1)
ax.set_title('Coefficient of Variation (%) by Algorithm and Device', fontsize=14, fontweight='bold')
ax.set_xlabel('Device')
ax.set_ylabel('Algorithm')
plt.tight_layout()
save_figure(fig, 'cv_heatmap_by_algorithm_device.png')

display_interpretation(
    'This heatmap shows CV values across all algorithm-device combinations. Darker colors (green) indicate lower and more desirable CV values, '
    'meaning more consistent performance. Brighter colors (red) indicate higher variability. This allows quick visual identification of '
    'which algorithm-device combinations offer the most stable and predictable performance.'
)

print('Statistical analysis plots created successfully.')


## 8.7 Advanced Analytical Visualizations: Performance Metrics and Correlations

Additional plots for deeper performance analysis, correlation insights, and multi-dimensional comparisons.

In [ ]:
# Advanced Performance Analysis Plots
print("Generating Advanced Analytical Visualizations...\n")

# 1. ALGORITHM PERFORMANCE RANKING (Multi-metric visualization)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1a. Speed Ranking
speed_ranking = benchmark_data.groupby('algorithm')['total_time_ms'].mean().sort_values()
sns.barplot(x=speed_ranking.values, y=speed_ranking.index, ax=axes[0,0], palette='viridis', orient='h')
axes[0,0].set_title('Speed Ranking: Average Total Time (ms)', fontweight='bold', fontsize=12)
axes[0,0].set_xlabel('Time (ms)')
axes[0,0].grid(axis='x', alpha=0.3)

# 1b. Energy Ranking
energy_ranking = benchmark_data.groupby('algorithm')['energy_J'].mean().sort_values()
sns.barplot(x=energy_ranking.values, y=energy_ranking.index, ax=axes[0,1], palette='plasma', orient='h')
axes[0,1].set_title('Energy Efficiency Ranking: Average Energy (J)', fontweight='bold', fontsize=12)
axes[0,1].set_xlabel('Energy (J)')
axes[0,1].grid(axis='x', alpha=0.3)

# 1c. Stability Ranking (by Time CV)
stability_ranking = aggregated_results.groupby('algorithm')['total_time_cv_pct'].mean().sort_values()
sns.barplot(x=stability_ranking.values, y=stability_ranking.index, ax=axes[1,0], palette='coolwarm', orient='h')
axes[1,0].set_title('Stability Ranking: Average Time CV (%)', fontweight='bold', fontsize=12)
axes[1,0].set_xlabel('Coefficient of Variation (%)')
axes[1,0].grid(axis='x', alpha=0.3)

# 1d. Throughput Ranking (KB/ms)
throughput_ranking = benchmark_data.groupby('algorithm')['throughput_kb_per_ms'].mean().sort_values(ascending=False)
sns.barplot(x=throughput_ranking.values, y=throughput_ranking.index, ax=axes[1,1], palette='RdYlGn', orient='h')
axes[1,1].set_title('Throughput Ranking: Average KB/ms', fontweight='bold', fontsize=12)
axes[1,1].set_xlabel('Throughput (KB/ms)')
axes[1,1].grid(axis='x', alpha=0.3)

fig.suptitle('Multi-Metric Algorithm Performance Rankings', fontsize=14, fontweight='bold', y=0.995)
fig.tight_layout()
save_figure(fig, 'algorithm_ranking_multi_metric.png')

display_interpretation(
    'This multi-panel ranking visualizes four independent performance dimensions: speed (time), energy efficiency, consistency (CV), '
    'and throughput. An ideal algorithm ranks high on all metrics; however, most algorithms show specialization (e.g., ElGamal excels in energy but ranks low in speed).'
)

# 2. CORRELATION HEATMAP: Time, Energy, Memory, Throughput
fig, ax = plt.subplots(figsize=(10, 8))
correlation_data = benchmark_data[['total_time_ms', 'energy_J', 'total_mem_bytes', 'throughput_kb_per_ms', 'size_bytes']].corr()
sns.heatmap(correlation_data, annot=True, fmt='.2f', cmap='coolwarm', center=0, cbar_kws={'label': 'Correlation'}, 
            square=True, ax=ax, linewidths=1, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix: Performance Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
save_figure(fig, 'correlation_metrics_heatmap.png')

display_interpretation(
    'This correlation matrix reveals relationships between performance dimensions. Strong positive correlation (red) indicates that metrics '
    'move together, while negative correlation (blue) indicates trade-offs. For example, strong correlation between time and energy suggests that '
    'fast algorithms also consume less energy, supporting their selection for resource-constrained environments.'
)

# 3. MEMORY USAGE ANALYSIS
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 3a. Memory by Algorithm
memory_by_algo = benchmark_data.groupby('algorithm')['total_mem_bytes'].mean().sort_values(ascending=False)
sns.barplot(data=benchmark_data, x='algorithm', y='total_mem_bytes', ax=axes[0], palette='Spectral', order=memory_by_algo.index)
axes[0].set_title('Average Memory Usage by Algorithm', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Algorithm')
axes[0].set_ylabel('Total Memory (bytes)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_yscale('log')

# 3b. Memory vs Time (scatter colored by size)
scatter_mem = axes[1].scatter(
    benchmark_data['total_time_ms'],
    benchmark_data['total_mem_bytes'],
    c=np.log10(benchmark_data['size_bytes']),
    s=50,
    alpha=0.6,
    cmap='viridis',
    edgecolors='black',
    linewidth=0.5
)
axes[1].set_xlabel('Total Time (ms)')
axes[1].set_ylabel('Total Memory (bytes)')
axes[1].set_title('Memory Usage vs Execution Time', fontweight='bold', fontsize=12)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter_mem, ax=axes[1])
cbar.set_label('Input Size\n(log10 bytes)')

fig.suptitle('Memory Footprint Analysis', fontsize=14, fontweight='bold', y=1.00)
fig.tight_layout()
save_figure(fig, 'memory_usage_analysis.png')

display_interpretation(
    'Memory analysis reveals the resource footprint of each algorithm. The scatter plot shows that some algorithms maintain constant memory '
    'regardless of input size (horizontal lines), while others scale linearly (diagonal). This is critical for embedded and mobile deployment '
    'where memory constraints are stringent.'
)

# 4. SCALING BEHAVIOR ANALYSIS: Input Size Impact
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prepare data for scaling analysis
scaling_data = aggregated_results[aggregated_results['algorithm'].isin(['AES-GCM', 'ChaCha20', 'ElGamal', 'RSA-Hybrid', 'ASCON'])].copy()

# 4a. Time scaling (log-log to detect power law)
for algo in scaling_data['algorithm'].unique():
    algo_data = scaling_data[scaling_data['algorithm'] == algo].sort_values('size_bytes')
    axes[0].loglog(algo_data['size_bytes'], algo_data['total_time_ms_mean'], marker='o', label=algo, linewidth=2)

axes[0].set_title('Execution Time Scaling (Log-Log Plot)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Input Size (bytes, log scale)')
axes[0].set_ylabel('Total Time (ms, log scale)')
axes[0].legend(loc='upper left', frameon=True)
axes[0].grid(True, alpha=0.3, which='both')

# 4b. Energy scaling
for algo in scaling_data['algorithm'].unique():
    algo_data = scaling_data[scaling_data['algorithm'] == algo].sort_values('size_bytes')
    axes[1].loglog(algo_data['size_bytes'], algo_data['energy_J_mean'], marker='s', label=algo, linewidth=2)

axes[1].set_title('Energy Consumption Scaling (Log-Log Plot)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Input Size (bytes, log scale)')
axes[1].set_ylabel('Energy (J, log scale)')
axes[1].legend(loc='upper left', frameon=True)
axes[1].grid(True, alpha=0.3, which='both')

fig.suptitle('Scaling Behavior: Linear vs Superlinear Complexity', fontsize=14, fontweight='bold', y=1.00)
fig.tight_layout()
save_figure(fig, 'scaling_behavior_analysis.png')

display_interpretation(
    'Log-log plots reveal the scaling exponent of each algorithm. A slope of 1 indicates linear scaling (ideal), while steeper slopes '
    'indicate superlinear or quadratic scaling. This helps predict behavior at larger input sizes beyond the benchmark range. '
    'For example, AES-GCM shows near-linear scaling, while Elephant shows steeper scaling curves.'
)

# 5. KERNEL DENSITY ESTIMATION (KDE) - Distribution Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 5a. Time KDE by Device
for device in benchmark_data['device'].unique():
    data_subset = benchmark_data[benchmark_data['device'] == device]['total_time_ms']
    data_subset.plot.kde(ax=axes[0], label=device, linewidth=2)

axes[0].set_title('Time Distribution KDE by Device', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Total Time (ms)')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 5b. Top 5 algorithms KDE
top_5_algos = benchmark_data.groupby('algorithm')['total_time_ms'].mean().nsmallest(5).index
for algo in top_5_algos:
    data_subset = benchmark_data[benchmark_data['algorithm'] == algo]['total_time_ms']
    data_subset.plot.kde(ax=axes[1], label=algo, linewidth=2)

axes[1].set_title('Time Distribution KDE: Top 5 Fastest Algorithms', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Total Time (ms)')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Kernel Density Estimation: Smooth Distribution Comparison', fontsize=14, fontweight='bold', y=1.00)
fig.tight_layout()
save_figure(fig, 'kde_distribution_comparison.png')

display_interpretation(
    'KDE (Kernel Density Estimation) provides smooth probability density estimates. Comparing KDE curves reveals whether distributions '
    'are unimodal (single peak), bimodal (two peaks), or heavy-tailed. The left plot compares devices; the right compares top algorithms. '
    'Wider, flatter curves indicate higher variability; narrow, tall peaks indicate consistent performance.'
)

# 6. PARETO FRONTIER - Time vs Energy Trade-off
fig, ax = plt.subplots(figsize=(14, 8))

# Calculate average metrics per algorithm
algo_summary = benchmark_data.groupby('algorithm').agg({
    'total_time_ms': 'mean',
    'energy_J': 'mean'
}).reset_index()

# Color code by rank
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(algo_summary)))

scatter = ax.scatter(algo_summary['total_time_ms'], algo_summary['energy_J'], 
                     s=500, alpha=0.7, c=range(len(algo_summary)), cmap='RdYlGn_r', 
                     edgecolors='black', linewidth=2)

# Annotate each point
for idx, row in algo_summary.iterrows():
    ax.annotate(row['algorithm'], 
                (row['total_time_ms'], row['energy_J']),
                xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')

# Identify and highlight Pareto optimal points (none worse in both dimensions)
def is_pareto_efficient(costs):
    is_efficient = np.ones(costs.shape[0], dtype=bool)
    for i, c in enumerate(costs):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(costs[is_efficient] <= c, axis=1)
            is_efficient[i] = True
    return is_efficient

pareto_coords = algo_summary[['total_time_ms', 'energy_J']].values
pareto_mask = is_pareto_efficient(pareto_coords)
pareto_algos = algo_summary[pareto_mask]

if len(pareto_algos) > 0:
    ax.scatter(pareto_algos['total_time_ms'], pareto_algos['energy_J'], 
              s=600, facecolors='none', edgecolors='red', linewidth=3, label='Pareto Frontier')

ax.set_xlabel('Average Total Time (ms)', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Energy (J)', fontsize=12, fontweight='bold')
ax.set_title('Pareto Frontier: Time-Energy Trade-off Analysis', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')
ax.legend(fontsize=11)

# Add quadrant annotations
ax.text(0.95, 0.95, 'IDEAL\n(Fast + Efficient)', transform=ax.transAxes, 
        ha='right', va='top', fontsize=10, style='italic', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax.text(0.05, 0.05, 'POOR\n(Slow + Inefficient)', transform=ax.transAxes, 
        ha='left', va='bottom', fontsize=10, style='italic', bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.3))

plt.tight_layout()
save_figure(fig, 'pareto_frontier_time_energy.png')

display_interpretation(
    'The Pareto frontier identifies algorithms that are not dominated in both metrics. Algorithms on the frontier represent optimal choices '
    'for different priorities: leftmost = fastest, bottommost = most energy-efficient. Algorithms inside the frontier (non-Pareto) are '
    'dominated by at least one frontier algorithm in both dimensions. Red circle highlights the Pareto frontier boundary.'
)

print('\n✓ All 6 advanced analytical plots generated successfully!')
print('Plots generated:')
print('  1. algorithm_ranking_multi_metric.png - Speed, Energy, Stability, Throughput')
print('  2. correlation_metrics_heatmap.png - Metric correlations')
print('  3. memory_usage_analysis.png - Memory footprint by algorithm')
print('  4. scaling_behavior_analysis.png - Time/Energy vs Input Size (log-log)')
print('  5. kde_distribution_comparison.png - Smooth distribution curves')
print('  6. pareto_frontier_time_energy.png - Pareto optimal algorithms')


In [ ]:
# 7. PERFORMANCE HEATMAP: Algorithm x Input Size Matrix
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# 7a. Time heatmap
time_pivot = aggregated_results.pivot_table(values='total_time_ms_mean', index='algorithm', columns='size_bytes', aggfunc='mean')
sns.heatmap(time_pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[0], cbar_kws={'label': 'Time (ms)'}, linewidths=0.5)
axes[0].set_title('Execution Time Heatmap: Algorithm × Input Size', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Input Size (bytes)')
axes[0].set_ylabel('Algorithm')

# 7b. Energy heatmap
energy_pivot = aggregated_results.pivot_table(values='energy_J_mean', index='algorithm', columns='size_bytes', aggfunc='mean')
sns.heatmap(energy_pivot, annot=True, fmt='.2e', cmap='RdPu', ax=axes[1], cbar_kws={'label': 'Energy (J)'}, linewidths=0.5)
axes[1].set_title('Energy Consumption Heatmap: Algorithm × Input Size', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Input Size (bytes)')
axes[1].set_ylabel('Algorithm')

fig.suptitle('Performance Matrix: Algorithm Behavior Across Input Sizes', fontsize=14, fontweight='bold', y=1.00)
fig.tight_layout()
save_figure(fig, 'performance_matrix_heatmap.png')

display_interpretation(
    'These heatmaps provide a comprehensive view of how each algorithm performs across the entire input size range. '
    'Darker colors indicate worse performance. Horizontal patterns suggest algorithm-independent behavior; '
    'diagonal patterns indicate scaling effects. This visualization helps identify "sweet spots" for each algorithm.'
)

# 8. SPEED vs STABILITY SCATTER: Which algorithm offers best consistency?
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    aggregated_results.groupby('algorithm')['total_time_ms_mean'].mean(),
    aggregated_results.groupby('algorithm')['total_time_cv_pct'].mean(),
    s=aggregated_results.groupby('algorithm')['energy_J_mean'].mean() * 10000,  # size = energy
    alpha=0.6,
    c=aggregated_results.groupby('algorithm')['throughput_kb_per_ms'].mean(),
    cmap='viridis',
    edgecolors='black',
    linewidth=2
)

# Annotate points
for algo in aggregated_results['algorithm'].unique():
    algo_data = aggregated_results[aggregated_results['algorithm'] == algo]
    x = algo_data['total_time_ms_mean'].mean()
    y = algo_data['total_time_cv_pct'].mean()
    ax.annotate(algo, (x, y), xytext=(5, 5), textcoords='offset points', fontsize=9, fontweight='bold')

ax.set_xlabel('Average Time (ms) - Lower is Better →', fontsize=12, fontweight='bold')
ax.set_ylabel('Time Stability (CV %) - Lower is Better ↓', fontsize=12, fontweight='bold')
ax.set_title('Speed vs Stability Trade-off (Bubble = Energy, Color = Throughput)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Throughput (KB/ms)')

# Add ideal region annotation
ax.axhspan(0, 15, alpha=0.1, color='green', label='Ideal Stability Zone (CV<15%)')
ax.legend(loc='upper right')

plt.tight_layout()
save_figure(fig, 'speed_vs_stability_analysis.png')

display_interpretation(
    'This bubble chart compares three dimensions simultaneously: X-axis = speed (lower = better), Y-axis = consistency/stability (lower = better), '
    'bubble size = energy consumption, and color = throughput. The ideal algorithm clusters in the lower-left corner. '
    'This identifies which algorithms offer the best combination of speed AND consistency.'
)

# 9. ALGORITHM PERFORMANCE BY DEVICE - Relative Performance Index
fig, ax = plt.subplots(figsize=(14, 7))

# Calculate performance index for each algorithm on each device (lower is better)
device_performance = benchmark_data.groupby(['device', 'algorithm']).agg({
    'total_time_ms': 'mean',
    'energy_J': 'mean'
}).reset_index()

# Normalize and combine
device_perf_pivot_time = device_performance.pivot(index='algorithm', columns='device', values='total_time_ms')
device_perf_pivot_energy = device_performance.pivot(index='algorithm', columns='device', values='energy_J')

# Create combined performance score (lower = better)
performance_index = (
    (device_perf_pivot_time / device_perf_pivot_time.max().max() * 100) +
    (device_perf_pivot_energy / device_perf_pivot_energy.max().max() * 100)
) / 2

performance_index_sorted = performance_index.sort_values(by=performance_index.columns[0])

# Stacked or grouped bar chart
x = np.arange(len(performance_index_sorted))
width = 0.35

for i, device in enumerate(performance_index_sorted.columns):
    ax.bar(x + i * width, performance_index_sorted[device], width, label=device, alpha=0.8)

ax.set_xlabel('Algorithm', fontweight='bold', fontsize=12)
ax.set_ylabel('Performance Index Score (Lower = Better)', fontweight='bold', fontsize=12)
ax.set_title('Device Comparison: Relative Performance Index by Algorithm', fontweight='bold', fontsize=14)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(performance_index_sorted.index, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_figure(fig, 'device_performance_comparison.png')

display_interpretation(
    'This chart compares algorithm performance across devices using a unified performance index that combines speed and energy efficiency. '
    'Lower scores are better. Algorithms with consistent bar heights across devices are portable; those with divergent heights are device-specific. '
    'This helps identify which algorithms are best for cross-platform deployment.'
)

# 10. PERCENTILE DISTRIBUTION - What about worst-case scenarios?
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

top_4_algos = benchmark_data.groupby('algorithm')['total_time_ms'].mean().nsmallest(4).index

for idx, algo in enumerate(top_4_algos):
    ax = axes[idx // 2, idx % 2]
    algo_data = benchmark_data[benchmark_data['algorithm'] == algo]['total_time_ms']
    
    # Calculate percentiles
    percentiles = np.arange(0, 101, 10)
    values = np.percentile(algo_data, percentiles)
    
    ax.plot(percentiles, values, marker='o', linewidth=2, markersize=8, color='#2E86AB')
    ax.fill_between(percentiles, values, alpha=0.3, color='#2E86AB')
    
    ax.set_title(f'{algo}: Percentile Distribution', fontweight='bold', fontsize=12)
    ax.set_xlabel('Percentile')
    ax.set_ylabel('Execution Time (ms)')
    ax.grid(True, alpha=0.3)
    
    # Add statistics
    mean_val = algo_data.mean()
    median_val = algo_data.median()
    p95_val = np.percentile(algo_data, 95)
    p99_val = np.percentile(algo_data, 99)
    
    stats_text = f'Mean: {mean_val:.2f}ms\nMedian: {median_val:.2f}ms\nP95: {p95_val:.2f}ms\nP99: {p99_val:.2f}ms'
    ax.text(0.98, 0.02, stats_text, transform=ax.transAxes, fontsize=9,
            verticalalignment='bottom', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('Percentile Analysis: Understanding Tail Latency (Worst-Case Scenarios)', fontsize=14, fontweight='bold', y=0.995)
fig.tight_layout()
save_figure(fig, 'percentile_distribution_analysis.png')

display_interpretation(
    'Percentile curves show how performance degrades at the tails. While mean time is important, tail latencies (P95, P99) are critical for QoS. '
    'For example, an algorithm with low mean but high P99 is unreliable. This visualization identifies algorithms with consistent tail performance. '
    'Flat curves indicate predictable performance; steep curves indicate occasional slowdowns.'
)

print('\n✓ Advanced analysis plots generated! (Plots 7-10)')
print('  7. performance_matrix_heatmap.png - Algorithm × Input Size matrix')
print('  8. speed_vs_stability_analysis.png - Multi-dimensional scatter')
print('  9. device_performance_comparison.png - Cross-device performance')
print('  10. percentile_distribution_analysis.png - Tail latency analysis')


## 8.8 ElGamal Deep Dive Analysis: Energy vs Time Trade-off Investigation

Specialized analysis focusing on ElGamal's unique position as the most energy-efficient algorithm despite having the longest execution time. This section investigates the trade-offs and applicability of ElGamal for battery-constrained deployments.

In [ ]:
# ELGAMAL DEEP ANALYSIS
print("\n" + "="*80)
print("ELGAMAL ALGORITHM: COMPREHENSIVE TRADE-OFF ANALYSIS")
print("="*80)

elgamal_data = benchmark_data[benchmark_data['algorithm'] == 'ElGamal'].copy()

# Key metrics for ElGamal
elg_time_mean = elgamal_data['total_time_ms'].mean()
elg_time_std = elgamal_data['total_time_ms'].std()
elg_time_cv = (elg_time_std / elg_time_mean) * 100
elg_energy_mean = elgamal_data['energy_J'].mean()
elg_energy_std = elgamal_data['energy_J'].std()
elg_energy_cv = (elg_energy_std / elg_energy_mean) * 100

# Comparison: ElGamal vs other algorithms
all_algos_summary = benchmark_data.groupby('algorithm').agg({
    'total_time_ms': ['mean', 'std'],
    'energy_J': ['mean', 'std']
}).reset_index()
all_algos_summary.columns = ['algorithm', 'time_mean', 'time_std', 'energy_mean', 'energy_std']

# Calculate ranking
all_algos_summary['time_rank'] = all_algos_summary['time_mean'].rank()
all_algos_summary['energy_rank'] = all_algos_summary['energy_mean'].rank()

elg_row = all_algos_summary[all_algos_summary['algorithm'] == 'ElGamal'].iloc[0]
fastest = all_algos_summary[all_algos_summary['time_mean'] == all_algos_summary['time_mean'].min()].iloc[0]
most_efficient = all_algos_summary[all_algos_summary['energy_mean'] == all_algos_summary['energy_mean'].min()].iloc[0]

time_penalty = (elg_row['time_mean'] / fastest['time_mean']) 
energy_gain = (fastest['energy_mean'] / elg_row['energy_mean']) if elg_row['energy_mean'] > 0 else 0

print(f"\nElGamal Performance Metrics:")
print(f"  Average Time: {elg_time_mean:.2f} ms (±{elg_time_std:.2f} ms, CV={elg_time_cv:.1f}%)")
print(f"  Average Energy: {elg_energy_mean:.6f} J (±{elg_energy_std:.6f} J, CV={elg_energy_cv:.1f}%)")
print(f"\nElGamal Trade-offs:")
print(f"  Time Penalty: {time_penalty:.2f}× slower than {fastest['algorithm']} (fastest)")
print(f"  Energy Gain: {energy_gain:.2f}× more efficient than {fastest['algorithm']}")
print(f"  Most efficient algorithm: {most_efficient['algorithm']} ({most_efficient['energy_mean']:.6f} J)")

# 1. ElGamal vs AES-GCM Comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1a. Time by Input Size
aesgcm_data = benchmark_data[benchmark_data['algorithm'] == 'AES-GCM'].copy()
for data, label, color in [(elgamal_data, 'ElGamal', '#d62728'), (aesgcm_data, 'AES-GCM', '#2ca02c')]:
    grouped = data.groupby('size_bytes')['total_time_ms'].agg(['mean', 'std']).reset_index()
    axes[0,0].errorbar(grouped['size_bytes'], grouped['mean'], yerr=grouped['std'], 
                       marker='o', label=label, linewidth=2, markersize=8, capsize=5, color=color)

axes[0,0].set_xlabel('Input Size (bytes)', fontweight='bold')
axes[0,0].set_ylabel('Execution Time (ms)', fontweight='bold')
axes[0,0].set_title('Time Comparison: ElGamal vs AES-GCM', fontweight='bold', fontsize=12)
axes[0,0].set_xscale('log')
axes[0,0].set_yscale('log')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3, which='both')

# 1b. Energy by Input Size
for data, label, color in [(elgamal_data, 'ElGamal', '#d62728'), (aesgcm_data, 'AES-GCM', '#2ca02c')]:
    grouped = data.groupby('size_bytes')['energy_J'].agg(['mean', 'std']).reset_index()
    axes[0,1].errorbar(grouped['size_bytes'], grouped['mean'], yerr=grouped['std'], 
                       marker='s', label=label, linewidth=2, markersize=8, capsize=5, color=color)

axes[0,1].set_xlabel('Input Size (bytes)', fontweight='bold')
axes[0,1].set_ylabel('Energy (J)', fontweight='bold')
axes[0,1].set_title('Energy Comparison: ElGamal vs AES-GCM', fontweight='bold', fontsize=12)
axes[0,1].set_xscale('log')
axes[0,1].set_yscale('log')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3, which='both')

# 1c. Time Distribution Comparison
violin_data_list = [
    elgamal_data[elgamal_data['device'] == 'D1']['total_time_ms'],
    elgamal_data[elgamal_data['device'] == 'D2']['total_time_ms'],
    aesgcm_data[aesgcm_data['device'] == 'D1']['total_time_ms'],
    aesgcm_data[aesgcm_data['device'] == 'D2']['total_time_ms']
]
positions = [1, 1.5, 2.5, 3]
bp = axes[1,0].boxplot(violin_data_list, positions=positions, widths=0.4, patch_artist=True,
                        labels=['EG-D1', 'EG-D2', 'AES-D1', 'AES-D2'])
for patch, color in zip(bp['boxes'], ['#d62728', '#d62728', '#2ca02c', '#2ca02c']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

axes[1,0].set_ylabel('Total Time (ms)', fontweight='bold')
axes[1,0].set_title('Time Distribution: ElGamal vs AES-GCM by Device', fontweight='bold', fontsize=12)
axes[1,0].set_yscale('log')
axes[1,0].grid(True, alpha=0.3, axis='y')

# 1d. Energy Efficiency Ratio (Time vs Energy Trade-off)
elgamal_by_size = elgamal_data.groupby('size_bytes').agg({'total_time_ms': 'mean', 'energy_J': 'mean'}).reset_index()
aesgcm_by_size = aesgcm_data.groupby('size_bytes').agg({'total_time_ms': 'mean', 'energy_J': 'mean'}).reset_index()

# Normalize both
elgamal_by_size['time_norm'] = elgamal_by_size['total_time_ms'] / elgamal_by_size['total_time_ms'].max()
elgamal_by_size['energy_norm'] = elgamal_by_size['energy_J'] / elgamal_by_size['energy_J'].max()
elgamal_by_size['tradeoff_ratio'] = elgamal_by_size['time_norm'] / elgamal_by_size['energy_norm']

aesgcm_by_size['time_norm'] = aesgcm_by_size['total_time_ms'] / aesgcm_by_size['total_time_ms'].max()
aesgcm_by_size['energy_norm'] = aesgcm_by_size['energy_J'] / aesgcm_by_size['energy_J'].max()
aesgcm_by_size['tradeoff_ratio'] = aesgcm_by_size['time_norm'] / aesgcm_by_size['energy_norm']

axes[1,1].plot(elgamal_by_size['size_bytes'], elgamal_by_size['tradeoff_ratio'], 
              marker='o', label='ElGamal', linewidth=2, markersize=8, color='#d62728')
axes[1,1].plot(aesgcm_by_size['size_bytes'], aesgcm_by_size['tradeoff_ratio'], 
              marker='s', label='AES-GCM', linewidth=2, markersize=8, color='#2ca02c')
axes[1,1].set_xlabel('Input Size (bytes)', fontweight='bold')
axes[1,1].set_ylabel('Time/Energy Trade-off Ratio', fontweight='bold')
axes[1,1].set_title('Trade-off Ratio: Normalized Time vs Energy', fontweight='bold', fontsize=12)
axes[1,1].set_xscale('log')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3, which='both')

fig.suptitle('ElGamal Deep Dive: Energy Efficiency Leader with Performance Trade-off', 
            fontsize=14, fontweight='bold', y=0.995)
fig.tight_layout()
save_figure(fig, 'elgamal_detailed_comparison.png')

display_interpretation(
    'ElGamal demonstrates an interesting trade-off: it is the most energy-efficient algorithm (lowest J consumption) '
    'but the slowest in execution time. The left plots show this stark contrast: ElGamal scales faster in time than in energy. '
    'The trade-off ratio (right bottom) shows that for every unit of time increase, ElGamal gains significant energy savings compared to AES-GCM. '
    'This makes ElGamal ideal for battery-constrained systems where energy budget is limited and processing latency is tolerable (e.g., background encryption).'
)

# 2. Energy Efficiency Frontier: Which algorithms are Pareto-optimal for energy?
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 2a. Energy vs Time with Pareto frontier highlighted
algo_energy_time = benchmark_data.groupby('algorithm').agg({
    'total_time_ms': 'mean',
    'energy_J': 'mean'
}).reset_index()

# Identify Pareto frontier for ENERGY EFFICIENCY (minimize both time and energy)
pareto_costs = algo_energy_time[['total_time_ms', 'energy_J']].values
is_pareto = is_pareto_efficient(pareto_costs)

scatter1 = axes[0].scatter(algo_energy_time['total_time_ms'], 
                          algo_energy_time['energy_J'],
                          s=300, alpha=0.6, c='lightblue', edgecolors='black', linewidth=2)

# Highlight ElGamal
elg_idx = algo_energy_time[algo_energy_time['algorithm'] == 'ElGamal'].index[0]
axes[0].scatter(algo_energy_time.loc[elg_idx, 'total_time_ms'], 
               algo_energy_time.loc[elg_idx, 'energy_J'],
               s=500, facecolors='none', edgecolors='red', linewidth=3, label='ElGamal')

# Highlight Pareto frontier
pareto_points = algo_energy_time[is_pareto]
axes[0].scatter(pareto_points['total_time_ms'], 
               pareto_points['energy_J'],
               s=400, facecolors='none', edgecolors='green', linewidth=2, label='Pareto Frontier')

for idx, row in algo_energy_time.iterrows():
    axes[0].annotate(row['algorithm'], 
                    (row['total_time_ms'], row['energy_J']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)

axes[0].set_xlabel('Average Time (ms) - Lower is Better →', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Average Energy (J) - Lower is Better ↓', fontweight='bold', fontsize=11)
axes[0].set_title('Energy-Time Trade-off Space: Pareto Frontier', fontweight='bold', fontsize=12)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].grid(True, alpha=0.3, which='both')

# Add region annotations
axes[0].text(0.98, 0.98, 'IDEAL\nFast + Efficient', transform=axes[0].transAxes, 
            ha='right', va='top', fontsize=9, style='italic', 
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
axes[0].text(0.02, 0.02, 'POOR\nSlow + Inefficient', transform=axes[0].transAxes, 
            ha='left', va='bottom', fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

# 2b. ElGamal Positioning Chart
positioning_data = benchmark_data.groupby('algorithm').agg({
    'total_time_ms': 'mean',
    'energy_J': 'mean',
    'throughput_kb_per_ms': 'mean'
}).reset_index()

positioning_data['energy_rank'] = positioning_data['energy_J'].rank()
positioning_data['speed_rank'] = positioning_data['total_time_ms'].rank(ascending=False)  # higher rank = faster

scatter2 = axes[1].scatter(positioning_data['speed_rank'], 
                          positioning_data['energy_rank'],
                          s=400, alpha=0.6, c='lightblue', edgecolors='black', linewidth=2)

# Highlight ElGamal
elg_idx2 = positioning_data[positioning_data['algorithm'] == 'ElGamal'].index[0]
axes[1].scatter(positioning_data.loc[elg_idx2, 'speed_rank'], 
               positioning_data.loc[elg_idx2, 'energy_rank'],
               s=500, facecolors='none', edgecolors='red', linewidth=3, marker='D', label='ElGamal')

# Highlight AES-GCM (best overall)
aes_idx = positioning_data[positioning_data['algorithm'] == 'AES-GCM'].index[0]
axes[1].scatter(positioning_data.loc[aes_idx, 'speed_rank'], 
               positioning_data.loc[aes_idx, 'energy_rank'],
               s=500, facecolors='none', edgecolors='green', linewidth=3, marker='*', label='AES-GCM (Best Overall)')

for idx, row in positioning_data.iterrows():
    axes[1].annotate(row['algorithm'], 
                    (row['speed_rank'], row['energy_rank']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)

axes[1].set_xlabel('Speed Ranking (Higher = Faster) →', fontweight='bold', fontsize=11)
axes[1].set_ylabel('Energy Efficiency Ranking (Lower = More Efficient) ↓', fontweight='bold', fontsize=11)
axes[1].set_title('Algorithm Positioning: Speed vs Energy Efficiency', fontweight='bold', fontsize=12)
axes[1].invert_yaxis()  # Invert y-axis so lower energy is up
axes[1].set_xlim(0, len(positioning_data) + 1)
axes[1].set_ylim(len(positioning_data) + 1, 0)
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper left', fontsize=10)

# Add quadrant lines
mid_x = (positioning_data['speed_rank'].min() + positioning_data['speed_rank'].max()) / 2
mid_y = (positioning_data['energy_rank'].min() + positioning_data['energy_rank'].max()) / 2
axes[1].axvline(mid_x, color='gray', linestyle='--', alpha=0.5, linewidth=1)
axes[1].axhline(mid_y, color='gray', linestyle='--', alpha=0.5, linewidth=1)

fig.suptitle('ElGamal Position Analysis: Trade-off Champion', fontsize=14, fontweight='bold', y=0.995)
fig.tight_layout()
save_figure(fig, 'elgamal_positioning_analysis.png')

display_interpretation(
    'The left plot shows ElGamal\'s unique position: it is the ONLY algorithm in the upper-left region (low energy but high time). '
    'The right plot uses ranking to show relative positioning. ElGamal ranks 1st in energy efficiency but last in speed. '
    'This specialization makes it valuable for specific use cases: battery-critical background operations, IoT devices with loose latency requirements, '
    'and scenarios where energy budget is the primary constraint rather than processing latency.'
)

# 3. Battery Life Simulation
fig, ax = plt.subplots(figsize=(14, 7))

# Assume different battery capacities and calculate approximate usage time
battery_capacities = np.array([1000, 2000, 5000, 10000])  # mWh (example values)
operations_per_battery = battery_capacities / (benchmark_data.groupby('algorithm')['energy_J'].mean() * 1000)  # Convert J to mWh

# For ElGamal vs AES-GCM
elg_ops = battery_capacities / (elgamal_data['energy_J'].mean() * 1000)
aes_ops = battery_capacities / (aesgcm_data['energy_J'].mean() * 1000)

x = np.arange(len(battery_capacities))
width = 0.35

bars1 = ax.bar(x - width/2, elg_ops, width, label='ElGamal', color='#d62728', alpha=0.8)
bars2 = ax.bar(x + width/2, aes_ops, width, label='AES-GCM', color='#2ca02c', alpha=0.8)

ax.set_xlabel('Battery Capacity (mWh)', fontweight='bold', fontsize=12)
ax.set_ylabel('Number of Operations Possible', fontweight='bold', fontsize=12)
ax.set_title('Battery Life Simulation: Operations Per Charge\n(1KB payload per operation)', 
            fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels([f'{int(c)}' for c in battery_capacities])
ax.legend(fontsize=11)
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y', which='both')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{int(height)}',
               ha='center', va='bottom', fontsize=9)

# Add efficiency gain annotation
efficiency_gain = (elg_ops[2] / aes_ops[2]) - 1
ax.text(0.98, 0.02, f'ElGamal enables {efficiency_gain*100:.0f}% more operations\non same battery',
       transform=ax.transAxes, ha='right', va='bottom', fontsize=11, fontweight='bold',
       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout()
save_figure(fig, 'elgamal_battery_simulation.png')

display_interpretation(
    'This simulation shows practical battery life implications. With a 5000mWh battery (typical smartphone), '
    f'ElGamal enables {elg_ops[2]:.0f} operations compared to {aes_ops[2]:.0f} with AES-GCM. '
    'For IoT devices or wearables operating on small batteries, choosing ElGamal can extend operational lifetime significantly. '
    'However, this comes at the cost of processing latency, so use cases must tolerate 6-10 second delays per encryption operation.'
)

print('\n✓ ElGamal deep analysis complete!')
print('Plots generated:')
print('  - elgamal_detailed_comparison.png')
print('  - elgamal_positioning_analysis.png')
print('  - elgamal_battery_simulation.png')


In [ ]:
overall_by_algorithm = (
    benchmark_data
    .groupby('algorithm', as_index=False)
    .agg(
        total_time_ms_mean=('total_time_ms', 'mean'),
        total_time_ms_median=('total_time_ms', 'median'),
        total_time_ms_std=('total_time_ms', 'std'),
        energy_J_mean=('energy_J', 'mean'),
        energy_J_median=('energy_J', 'median'),
        energy_J_std=('energy_J', 'std'),
        rounds=('round_index', 'count')
    )
)

# Calculate Coefficient of Variation (%) for time and energy
overall_by_algorithm['total_time_cv_pct'] = (overall_by_algorithm['total_time_ms_std'] / overall_by_algorithm['total_time_ms_mean']) * 100
overall_by_algorithm['energy_cv_pct'] = (overall_by_algorithm['energy_J_std'] / overall_by_algorithm['energy_J_mean']) * 100

fastest_per_device = (
    benchmark_data.groupby(['device', 'algorithm'], as_index=False)
    .agg(total_time_ms_mean=('total_time_ms', 'mean'), energy_J_mean=('energy_J', 'mean'))
    .sort_values(['device', 'total_time_ms_mean'])
    .groupby('device', as_index=False)
    .first()
    .rename(columns={'algorithm': 'best_algorithm', 'total_time_ms_mean': 'best_total_time_ms_mean'})
)

lowest_energy_per_device = (
    benchmark_data.groupby(['device', 'algorithm'], as_index=False)
    .agg(total_time_ms_mean=('total_time_ms', 'mean'), energy_J_mean=('energy_J', 'mean'))
    .sort_values(['device', 'energy_J_mean'])
    .groupby('device', as_index=False)
    .first()
    .rename(columns={'algorithm': 'best_algorithm', 'energy_J_mean': 'best_energy_J_mean'})
)

best_per_input_size = (
    benchmark_data.groupby(['size_bytes', 'algorithm'], as_index=False)
    .agg(total_time_ms_mean=('total_time_ms', 'mean'), energy_J_mean=('energy_J', 'mean'))
    .sort_values(['size_bytes', 'total_time_ms_mean'])
    .groupby('size_bytes', as_index=False)
    .first()
    .rename(columns={'algorithm': 'best_algorithm', 'total_time_ms_mean': 'best_total_time_ms_mean'})
)

ranking_time = overall_by_algorithm.sort_values('total_time_ms_mean')
ranking_energy = overall_by_algorithm.sort_values('energy_J_mean')

summary_rankings = pd.concat([
    fastest_per_device.assign(ranking_type='fastest_per_device'),
    lowest_energy_per_device.assign(ranking_type='lowest_energy_per_device'),
    best_per_input_size.assign(ranking_type='best_per_input_size')
], ignore_index=True, sort=False)

summary_path = TABLES_DIR / 'summary_rankings.csv'
summary_rankings.to_csv(summary_path, index=False)

print('Fastest algorithm per device:')
display(fastest_per_device)
print('Lowest-energy algorithm per device:')
display(lowest_energy_per_device)
print('Best algorithm per input size:')
display(best_per_input_size)
print('Ranking by average time:')
display(ranking_time)
print('Ranking by average energy:')
display(ranking_energy)
print(f'Saved summary rankings to: {summary_path}')

Fastest algorithm per device:


,device,best_algorithm,best_total_time_ms_mean,energy_J_mean
0,D1,AES-GCM,6.3307,0.0004
1,D2,AES-GCM,6.2992,0.0005


Lowest-energy algorithm per device:


,device,best_algorithm,total_time_ms_mean,best_energy_J_mean
0,D1,ElGamal,38.5726,0.0002
1,D2,ElGamal,38.4249,0.0002


Best algorithm per input size:


...

Ranking by average time:


,algorithm,total_time_ms_mean,total_time_ms_median,total_time_ms_std,energy_J_mean,energy_J_median,energy_J_std,rounds
1,AES-GCM,6.3150,4.5796,4.4367,0.0005,0.0002,0.0005,90
3,ChaCha20,9.6171,5.1231,9.6617,0.0008,0.0003,0.0010,90
0,AES,12.1958,5.2467,13.4544,0.0015,0.0006,0.0018,90
2,ASCON,12.8408,8.6229,10.0258,0.0005,0.0002,0.0006,90
9,Xoodyak,16.2028,8.8625,15.3123,0.0006,0.0002,0.0007,90
8,RSA-Hybrid,37.9185,31.1956,13.6906,0.0002,0.0002,0.0001,90
4,ElGamal,38.4988,31.3004,13.2428,0.0002,0.0001,0.0001,90
6,GIFT-COFB,52.1101,25.1930,54.7027,0.0008,0.0003,0.0009,90
7,Grain-128AEAD,90.6167,33.2195,111.4152,0.0016,0.0005,0.0021,90
5,Elephant,225.9220,80.0766,287.2586,0.0022,0.0007,0.0029,90


Ranking by average energy:


,algorithm,total_time_ms_mean,total_time_ms_median,total_time_ms_std,energy_J_mean,energy_J_median,energy_J_std,rounds
4,ElGamal,38.4988,31.3004,13.2428,0.0002,0.0001,0.0001,90
8,RSA-Hybrid,37.9185,31.1956,13.6906,0.0002,0.0002,0.0001,90
1,AES-GCM,6.3150,4.5796,4.4367,0.0005,0.0002,0.0005,90
2,ASCON,12.8408,8.6229,10.0258,0.0005,0.0002,0.0006,90
9,Xoodyak,16.2028,8.8625,15.3123,0.0006,0.0002,0.0007,90
6,GIFT-COFB,52.1101,25.1930,54.7027,0.0008,0.0003,0.0009,90
3,ChaCha20,9.6171,5.1231,9.6617,0.0008,0.0003,0.0010,90
0,AES,12.1958,5.2467,13.4544,0.0015,0.0006,0.0018,90
7,Grain-128AEAD,90.6167,33.2195,111.4152,0.0016,0.0005,0.0021,90
5,Elephant,225.9220,80.0766,287.2586,0.0022,0.0007,0.0029,90


Saved summary rankings to: c:\Tese\results\benchmark_analysis_outputs\tables\summary_rankings.csv


## 10. Compute Normalized Trade-off Score

The normalized score combines runtime and energy into a single comparable measure. Lower values are better, and the weight alpha controls how much importance is given to time versus energy.

In [ ]:
tradeoff_scores = aggregated_results.copy()
tradeoff_scores['time_norm'] = min_max_normalize(tradeoff_scores['total_time_ms_mean'])
tradeoff_scores['energy_norm'] = min_max_normalize(tradeoff_scores['energy_J_mean'])
tradeoff_scores['score'] = ALPHA * tradeoff_scores['time_norm'] + (1 - ALPHA) * tradeoff_scores['energy_norm']

tradeoff_ranking = (
    tradeoff_scores.groupby('algorithm', as_index=False)
    .agg(
        score_mean=('score', 'mean'),
        score_median=('score', 'median'),
        score_std=('score', 'std'),
        total_time_ms_mean=('total_time_ms_mean', 'mean'),
        energy_J_mean=('energy_J_mean', 'mean'),
        total_time_cv_pct=('total_time_cv_pct', 'mean'),
        energy_cv_pct=('energy_cv_pct', 'mean')
    )
    .sort_values(['score_mean', 'score_std', 'total_time_ms_mean'])
    .reset_index(drop=True)
)

alpha_sensitivity = []
for alpha_value in [0.25, 0.5, 0.75]:
    scores = aggregated_results.copy()
    scores['time_norm'] = min_max_normalize(scores['total_time_ms_mean'])
    scores['energy_norm'] = min_max_normalize(scores['energy_J_mean'])
    scores['score'] = alpha_value * scores['time_norm'] + (1 - alpha_value) * scores['energy_norm']
    best = (
        scores.groupby('algorithm', as_index=False)['score']
        .mean()
        .sort_values('score')
        .iloc[0]
    )
    alpha_sensitivity.append({'alpha': alpha_value, 'best_algorithm': best['algorithm'], 'mean_score': best['score']})

alpha_sensitivity_table = pd.DataFrame(alpha_sensitivity)
best_overall_algorithm = tradeoff_ranking.iloc[0]

print('Alpha sensitivity:')
display(alpha_sensitivity_table)
print('Trade-off ranking:')
display(tradeoff_ranking)
print(f"Best overall algorithm at alpha={ALPHA}: {best_overall_algorithm['algorithm']}")
print(
    f"Reason: it has the lowest combined normalized score ({best_overall_algorithm['score_mean']:.4f}) "
    f"with a strong balance between average total time ({best_overall_algorithm['total_time_ms_mean']:.4f} ms) "
    f"and average energy ({best_overall_algorithm['energy_J_mean']:.6f} J)."
)

Alpha sensitivity:


,alpha,best_algorithm,mean_score
0,0.2500,ElGamal,0.0171
1,0.5000,AES-GCM,0.0207
2,0.7500,AES-GCM,0.0123


Trade-off ranking:


,algorithm,score_mean,score_median,score_std,total_time_ms_mean,energy_J_mean
0,AES-GCM,0.0207,0.0080,0.0272,6.3150,0.0005
1,ElGamal,0.0242,0.0176,0.0138,38.4988,0.0002
2,RSA-Hybrid,0.0249,0.0183,0.0137,37.9185,0.0002
3,ASCON,0.0275,0.0094,0.0358,12.8408,0.0005
4,Xoodyak,0.0341,0.0102,0.0481,16.2028,0.0006
5,ChaCha20,0.0386,0.0103,0.0570,9.6171,0.0008
6,GIFT-COFB,0.0613,0.0229,0.0780,52.1101,0.0008
7,AES,0.0774,0.0275,0.1048,12.1958,0.0015
8,Grain-128AEAD,0.1234,0.0356,0.1720,90.6167,0.0016
9,Elephant,0.2293,0.0731,0.3109,225.9220,0.0022


Best overall algorithm at alpha=0.5: AES-GCM
Reason: it has the lowest combined normalized score (0.0207) with a strong balance between average total time (6.3150 ms) and average energy (0.000458 J).


## 11. Compare PC Time Folder Results

The `time` folder contains the benchmark results from the PC run. This section compares the PC data against the mobile runs for the algorithms that appear in both datasets.

## 12. Identify Best Overall Algorithm

The best overall choice is selected from the trade-off ranking by combining normalized time and energy, then checking whether the result remains stable under nearby alpha values.

## 13. Export Tables and Figures

This final step verifies that the aggregated results, summary rankings, and saved figures are present in the export folders.

In [21]:
conclusion_text = (
    f"The composite ranking selects {best_overall_algorithm['algorithm']} as the best overall algorithm at alpha={ALPHA}. "
    'It stays near the top in average time while also remaining competitive on energy, which makes it the strongest balanced choice for a mobile benchmark study.'
)

display(Markdown(f'**Conclusion.** {conclusion_text}'))

tradeoff_path = TABLES_DIR / 'tradeoff_ranking.csv'
tradeoff_ranking.to_csv(tradeoff_path, index=False)

expected_exports = {
    'aggregated_results.csv': aggregated_path,
    'summary_rankings.csv': summary_path,
    'tradeoff_ranking.csv': tradeoff_path,
    'pc_algorithm_ranking.csv': pc_algorithm_path,
    'pc_vs_mobile_comparison.csv': pc_mobile_path,
    'time_trends_by_algorithm_device.png': PLOTS_DIR / 'time_trends_by_algorithm_device.png',
    'energy_memory_trends.png': PLOTS_DIR / 'energy_memory_trends.png',
    'algorithm_comparison_by_device.png': PLOTS_DIR / 'algorithm_comparison_by_device.png',
    'device_comparison_by_algorithm.png': PLOTS_DIR / 'device_comparison_by_algorithm.png',
    'round_level_variability_total_time.png': PLOTS_DIR / 'round_level_variability_total_time.png',
    'pc_vs_mobile_time.png': PLOTS_DIR / 'pc_vs_mobile_time.png',
    'pc_vs_mobile_energy.png': PLOTS_DIR / 'pc_vs_mobile_energy.png',
    'pc_vs_mobile_ratios.png': PLOTS_DIR / 'pc_vs_mobile_ratios.png',
    'pc_algorithm_summary.png': PLOTS_DIR / 'pc_algorithm_summary.png',
}

for device_name in sorted(benchmark_data['device'].dropna().unique()):
    expected_exports[f'device_{slugify(device_name)}_summary.png'] = PLOTS_DIR / f'device_{slugify(device_name)}_summary.png'

export_status = pd.DataFrame([
    {'artifact': name, 'exists': path.exists(), 'path': str(path)}
    for name, path in expected_exports.items()
])
display(export_status)
print(f"Export verification complete: {int(export_status['exists'].sum())}/{len(export_status)} artifacts found")

**Conclusion.** The composite ranking selects AES-GCM as the best overall algorithm at alpha=0.5. It stays near the top in average time while also remaining competitive on energy, which makes it the strongest balanced choice for a mobile benchmark study.

,artifact,exists,path
0,aggregated_results.csv,True,c:\Tese\results\benchmark_analysis_outputs\tab...
1,summary_rankings.csv,True,c:\Tese\results\benchmark_analysis_outputs\tab...
2,tradeoff_ranking.csv,True,c:\Tese\results\benchmark_analysis_outputs\tab...
3,pc_algorithm_ranking.csv,True,c:\Tese\results\benchmark_analysis_outputs\tab...
4,pc_vs_mobile_comparison.csv,True,c:\Tese\results\benchmark_analysis_outputs\tab...
5,time_trends_by_algorithm_device.png,True,c:\Tese\results\benchmark_analysis_outputs\plo...
6,energy_memory_trends.png,True,c:\Tese\results\benchmark_analysis_outputs\plo...
7,algorithm_comparison_by_device.png,True,c:\Tese\results\benchmark_analysis_outputs\plo...
8,device_comparison_by_algorithm.png,True,c:\Tese\results\benchmark_analysis_outputs\plo...
9,round_level_variability_total_time.png,True,c:\Tese\results\benchmark_analysis_outputs\plo...


Export verification complete: 16/16 artifacts found


In [20]:
pc_time_folder = ROOT_DIR / 'time'
pc_raw_files = sorted(pc_time_folder.glob('*_per_exec.csv'))

if not pc_raw_files:
    raise FileNotFoundError(f'No PC benchmark CSV files found under {pc_time_folder}')

pc_frames = []
pc_file_summary = []

for file_path in pc_raw_files:
    algorithm_name = pretty_label(normalize_filename_stem(file_path.stem))
    frame = pd.read_csv(file_path, usecols=['size_bytes', 'round_index', 'enc_ns', 'dec_ns', 'energy_mWh'])
    frame['algorithm'] = algorithm_name
    frame['device'] = 'PC'
    frame['platform'] = 'PC'
    frame['source_file'] = file_path.name
    frame['source_path'] = str(file_path)
    frame['enc_ms'] = frame['enc_ns'] / 1_000_000
    frame['dec_ms'] = frame['dec_ns'] / 1_000_000
    frame['total_time_ms'] = frame['enc_ms'] + frame['dec_ms']
    frame['energy_J'] = frame['energy_mWh'] * 3.6
    pc_frames.append(frame)
    pc_file_summary.append({'source_file': file_path.name, 'rows': len(frame), 'algorithm': algorithm_name})

pc_time_data = pd.concat(pc_frames, ignore_index=True)
pc_file_summary = pd.DataFrame(pc_file_summary)
common_algorithms = sorted(set(pc_time_data['algorithm']).intersection(set(benchmark_data['algorithm'])))

mobile_comparison = (
    benchmark_data[benchmark_data['algorithm'].isin(common_algorithms)]
    .groupby(['algorithm', 'size_bytes'], as_index=False)
    .agg(
        total_time_ms_mean=('total_time_ms', 'mean'),
        energy_J_mean=('energy_J', 'mean'),
        rounds=('round_index', 'count')
    )
)
mobile_comparison['platform'] = 'Mobile (D1/D2 average)'

pc_comparison = (
    pc_time_data[pc_time_data['algorithm'].isin(common_algorithms)]
    .groupby(['algorithm', 'size_bytes'], as_index=False)
    .agg(
        total_time_ms_mean=('total_time_ms', 'mean'),
        energy_J_mean=('energy_J', 'mean'),
        rounds=('round_index', 'count')
    )
)
pc_comparison['platform'] = 'PC'

pc_mobile_comparison = pd.concat([pc_comparison, mobile_comparison], ignore_index=True)

pc_algorithm_ranking = (
    pc_time_data.groupby('algorithm', as_index=False)
    .agg(
        total_time_ms_mean=('total_time_ms', 'mean'),
        total_time_ms_median=('total_time_ms', 'median'),
        energy_J_mean=('energy_J', 'mean'),
        energy_J_median=('energy_J', 'median'),
        rounds=('round_index', 'count')
    )
    .sort_values('total_time_ms_mean')
    .reset_index(drop=True)
)

pc_mobile_summary = (
    pc_comparison[['algorithm', 'size_bytes', 'total_time_ms_mean', 'energy_J_mean']]
    .merge(
        mobile_comparison[['algorithm', 'size_bytes', 'total_time_ms_mean', 'energy_J_mean']],
        on=['algorithm', 'size_bytes'],
        suffixes=('_pc', '_mobile')
    )
)
pc_mobile_summary['time_ratio_mobile_over_pc'] = pc_mobile_summary['total_time_ms_mean_mobile'] / pc_mobile_summary['total_time_ms_mean_pc']
pc_mobile_summary['energy_ratio_mobile_over_pc'] = pc_mobile_summary['energy_J_mean_mobile'] / pc_mobile_summary['energy_J_mean_pc']

pc_mobile_overview = (
    pc_mobile_summary.groupby('algorithm', as_index=False)
    .agg(
        pc_time_mean=('total_time_ms_mean_pc', 'mean'),
        mobile_time_mean=('total_time_ms_mean_mobile', 'mean'),
        pc_energy_mean=('energy_J_mean_pc', 'mean'),
        mobile_energy_mean=('energy_J_mean_mobile', 'mean'),
        mean_time_ratio=('time_ratio_mobile_over_pc', 'mean'),
        mean_energy_ratio=('energy_ratio_mobile_over_pc', 'mean')
    )
    .sort_values('mean_time_ratio', ascending=False)
)

pc_algorithm_path = TABLES_DIR / 'pc_algorithm_ranking.csv'
pc_algorithm_ranking.to_csv(pc_algorithm_path, index=False)

pc_mobile_path = TABLES_DIR / 'pc_vs_mobile_comparison.csv'
pc_mobile_overview.to_csv(pc_mobile_path, index=False)

# PC vs mobile time comparison with log scale on both axes.
time_compare_plot = sns.relplot(
    data=pc_mobile_comparison,
    x='size_bytes',
    y='total_time_ms_mean',
    hue='platform',
    style='platform',
    col='algorithm',
    kind='line',
    marker='o',
    facet_kws={'sharey': False, 'sharex': True},
    height=4.2,
    aspect=1.05,
)
time_compare_plot.set(xscale='log', yscale='log')
time_compare_plot.set_axis_labels('Input size (bytes, log2 scale)', 'Total time (ms, log scale)')
time_compare_plot.set_titles('{col_name}')
time_compare_plot.figure.suptitle('PC vs Mobile: Total Time by Algorithm', y=1.03)
time_compare_plot.figure.tight_layout()
time_compare_plot.figure.savefig(PLOTS_DIR / 'pc_vs_mobile_time.png', bbox_inches='tight', dpi=300)
plt.close(time_compare_plot.figure)

# PC vs mobile energy comparison with log scale on both axes.
energy_compare_plot = sns.relplot(
    data=pc_mobile_comparison,
    x='size_bytes',
    y='energy_J_mean',
    hue='platform',
    style='platform',
    col='algorithm',
    kind='line',
    marker='o',
    facet_kws={'sharey': False, 'sharex': True},
    height=4.2,
    aspect=1.05,
)
energy_compare_plot.set(xscale='log', yscale='log')
energy_compare_plot.set_axis_labels('Input size (bytes, log2 scale)', 'Energy (J, log scale)')
energy_compare_plot.set_titles('{col_name}')
energy_compare_plot.figure.suptitle('PC vs Mobile: Energy by Algorithm', y=1.03)
energy_compare_plot.figure.tight_layout()
energy_compare_plot.figure.savefig(PLOTS_DIR / 'pc_vs_mobile_energy.png', bbox_inches='tight', dpi=300)
plt.close(energy_compare_plot.figure)

# Ratio plot to make the scale gap explicit.
ratio_melted = pc_mobile_overview.melt(
    id_vars='algorithm',
    value_vars=['mean_time_ratio', 'mean_energy_ratio'],
    var_name='metric',
    value_name='ratio'
)
ratio_melted['metric'] = ratio_melted['metric'].map({
    'mean_time_ratio': 'Mobile / PC Time Ratio',
    'mean_energy_ratio': 'Mobile / PC Energy Ratio',
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
for metric_name, axis in zip(['Mobile / PC Time Ratio', 'Mobile / PC Energy Ratio'], axes):
    subset = ratio_melted[ratio_melted['metric'] == metric_name]
    sns.barplot(data=subset, x='algorithm', y='ratio', ax=axis, color='#4c72b0')
    axis.set_title(metric_name)
    axis.set_xlabel('Algorithm')
    axis.set_ylabel('Ratio')
    axis.tick_params(axis='x', rotation=35)
    axis.set_yscale('log')
    axis.axhline(1, color='black', linewidth=1, linestyle='--')

fig.suptitle('PC vs Mobile Ratio Summary', y=1.02)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'pc_vs_mobile_ratios.png', bbox_inches='tight', dpi=300)
plt.close(fig)

# PC-only summary figure to make the computer baseline explicit.
pc_summary_long = pc_algorithm_ranking.melt(
    id_vars='algorithm',
    value_vars=['total_time_ms_mean', 'energy_J_mean'],
    var_name='metric',
    value_name='value'
)
pc_summary_long['metric'] = pc_summary_long['metric'].map({
    'total_time_ms_mean': 'Mean Total Time (ms)',
    'energy_J_mean': 'Mean Energy (J)',
})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for metric_name, axis in zip(['Mean Total Time (ms)', 'Mean Energy (J)'], axes):
    subset = pc_summary_long[pc_summary_long['metric'] == metric_name]
    sns.barplot(data=subset, x='algorithm', y='value', ax=axis, color='#dd8452')
    axis.set_title(f'PC Baseline: {metric_name}')
    axis.set_xlabel('Algorithm')
    axis.set_ylabel(metric_name)
    axis.tick_params(axis='x', rotation=35)
    axis.set_yscale('log')

fig.suptitle('PC / Computer Baseline Summary', y=1.02)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'pc_algorithm_summary.png', bbox_inches='tight', dpi=300)
plt.close(fig)

display_interpretation('The PC curves provide a direct baseline for the same algorithms measured on the mobile devices. Log scaling makes the computer results visible even when the absolute values differ by an order of magnitude or more, and the ratio summary highlights how much slower or faster the mobile devices are relative to the PC.')

display(pc_file_summary)
display(pc_algorithm_ranking)
display(pc_mobile_overview)
print(f'Saved PC-only ranking to: {pc_algorithm_path}')
print(f'Saved PC-vs-mobile comparison table to: {pc_mobile_path}')

**Interpretation.** The PC curves provide a direct baseline for the same algorithms measured on the mobile devices. Log scaling makes the computer results visible even when the absolute values differ by an order of magnitude or more, and the ratio summary highlights how much slower or faster the mobile devices are relative to the PC.

,source_file,rows,algorithm
0,aes_bench_2p10_2p20_per_exec.csv,165,AES
1,aesgcm_bench_2p10_2p20_per_exec.csv,165,AES-GCM
2,chacha20_bench_2p10_2p20_per_exec.csv,165,ChaCha20
3,elgamal_bench_2p10_2p20_per_exec.csv,165,ElGamal
4,rsa_hybrid_bench_2p10_2p20_per_exec.csv,165,RSA-Hybrid


,algorithm,total_time_ms_mean,total_time_ms_median,energy_J_mean,energy_J_median,rounds
0,AES-GCM,8.3932,3.0085,0.0563,0.0311,165
1,ChaCha20,16.1282,3.1704,0.0922,0.0308,165
2,AES,18.3687,4.5416,0.1005,0.0309,165
3,RSA-Hybrid,18.5917,3.8235,0.1063,0.0308,165
4,ElGamal,18.9334,4.0876,0.1063,0.0311,165


,algorithm,pc_time_mean,mobile_time_mean,pc_energy_mean,mobile_energy_mean,mean_time_ratio,mean_energy_ratio
3,ElGamal,6.5430,38.4988,0.0512,0.0002,16.4337,0.0038
4,RSA-Hybrid,6.5872,37.9185,0.0514,0.0002,12.5595,0.0044
2,ChaCha20,5.3879,9.6171,0.0447,0.0008,3.1752,0.0134
1,AES-GCM,3.5351,6.3150,0.0344,0.0005,2.9836,0.0119
0,AES,6.5865,12.1958,0.0477,0.0015,1.8446,0.0238


Saved PC-only ranking to: c:\Tese\results\benchmark_analysis_outputs\tables\pc_algorithm_ranking.csv
Saved PC-vs-mobile comparison table to: c:\Tese\results\benchmark_analysis_outputs\tables\pc_vs_mobile_comparison.csv


In [18]:
device_png_paths = []
mobile_devices = sorted(benchmark_data['device'].dropna().unique())

for device_name in mobile_devices:
    device_subset = benchmark_data[benchmark_data['device'] == device_name].copy()
    if device_subset.empty:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

    sns.lineplot(
        data=device_subset,
        x='size_bytes',
        y='total_time_ms',
        hue='algorithm',
        marker='o',
        ax=axes[0],
    )
    sns.lineplot(
        data=device_subset,
        x='size_bytes',
        y='energy_J',
        hue='algorithm',
        marker='o',
        ax=axes[1],
        legend=False,
    )

    axes[0].set_title(f'{device_name}: Total Time by Algorithm')
    axes[1].set_title(f'{device_name}: Energy by Algorithm')
    for axis in axes:
        axis.set_xscale('log', base=2)
        axis.set_xlabel('Input size (bytes, log2 scale)')
    axes[0].set_ylabel('Total time (ms)')
    axes[1].set_ylabel('Energy (J)')

    legend = axes[0].get_legend()
    if legend is not None:
        legend.set_title('Algorithm')
        legend.set_bbox_to_anchor((1.02, 1))
        legend._loc = 2

    fig.suptitle(f'{device_name} Benchmark Summary', y=1.02)
    fig.tight_layout()
    output_path = PLOTS_DIR / f'device_{slugify(device_name)}_summary.png'
    fig.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.close(fig)
    device_png_paths.append(output_path)

print(f'Created {len(device_png_paths)} per-device PNG files.')
display(pd.DataFrame({'device': mobile_devices, 'png_path': [str(path) for path in device_png_paths]}))

Created 2 per-device PNG files.


,device,png_path
0,D1,c:\Tese\results\benchmark_analysis_outputs\plo...
1,D2,c:\Tese\results\benchmark_analysis_outputs\plo...
